# Track-Based Phagocytosis Analysis — across island sizes & replicates

Detects and quantifies macrophage phagocytosis of Raji B cells on micropatterned islands
(30 / 60 / 90 um), across replicates, using **two independent modes of analysis** and
their agreement. Colour convention in every image view: **red = Raji, blue = macrophage.**

**Mode 1 — Tracking analysis (sections 4-8b).** From the Cellpose + TrackMate spot tracks,
a *contact* is scored when a Raji centroid falls within `RADIUS_FACTOR` x (macrophage
radius) for >= `MIN_FRAMES` sustained frames (gap tolerance `GAP_TOL`). A contact is called
an *ingestion* when the Raji track disappears during / just after the overlap (`END_TOL`)
and is not censored at the final frame. Purely geometric: fast, but blind to whether the
red cell is truly *inside* the macrophage.

**Mode 2 — Signal colocalization (section 8c).** An independent pixel-level check on the
drift-corrected movies. At the macrophage centroid we measure **Manders' M1** — the
fraction of local Raji (red) signal lying inside the macrophage (blue) body — and call a
frame *colocalized* when M1 >= `COLOC_M1_MIN`. Applying the *same* sustained-frame limits
as the tracking (`MIN_FRAMES`, `GAP_TOL`), an event is *colocalization-confirmed* when the
red signal stays inside the macrophage long enough. This evidences physical engulfment
rather than mere proximity.

**Corroboration & filtering (section 8d).** Every candidate contact is classified by the
two modes into *tracking-only* (track vanished, no sustained colocalization),
*colocalization-only* (red inside the macrophage, track persisted), and
**double-corroborated** (both modes agree — the highest-confidence phagocytosis calls,
exported to `phago_double_corroborated_events.csv`). The events are then shown on the raw
movie image.

**Visual proof (section 8e).** A short in-page clip and a saved TIFF stack
(`phago_event_movie.tif`) of a double-corroborated event, plus a montage contrasting one
tracking-only, one colocalization-only, and one double-corroborated event.

Sections 9-12: notes, retuning / threshold sweeps, and manual-check snapshot tools.

## 0. Requirements, data layout & how to run

**Environment.** Python 3.13.5 with numpy 2.1.3, pandas 2.2.3, matplotlib 3.10.0,
scipy 1.15.3, scikit-image 0.25.0, tifffile 2025.2.18, statsmodels 0.14.4, and
jupyterlab 4.3.4. No ffmpeg or TIFF codec plugins are needed — the drift-corrected stacks
are uncompressed 8-bit TIFFs and the in-page clip uses matplotlib's JS writer.

```
pip install numpy==2.1.3 pandas==2.2.3 matplotlib==3.10.0 scipy==1.15.3 \
            scikit-image==0.25.0 tifffile==2025.2.18 statsmodels==0.14.4
```

**Data layout.** `DATA_ROOT` (set in the config cell below) must hold one folder per
`size × replicate`. Each folder carries both channels' TrackMate exports, both
drift-corrected movies, and a hand-drawn binary island mask:

```
<DATA_ROOT>/
  30 µm/R1/
    <stem>_spots.csv                  TrackMate spots, macrophage channel
    <stem>_xyCorrected.tif            drift-corrected stack, macrophage channel
    <stem>_xyCorrected.tif            drift-corrected stack, Raji channel
    *_spots_unfiltered.csv            TrackMate spots, Raji channel
    BCellM0_30um_R1_mask_whole.tif    island mask, on the movie's pixel grid
  30 µm/R3/  …  60 µm/R2/  …  90 µm/R5/
```

Two conventions matter, both because acquisition settings were not uniform:

- **Cell type is inferred from median spot radius** (Raji ≈ 6 µm, macrophage ≈ 13 µm), not
  from the channel number — channel numbering differs between acquisitions (`C=1`/`C=2` in
  some folders, `C2-`/`C3-` in others).
- **A movie is paired to its spots file by shared filename stem**
  (`<stem>_spots.csv` ↔ `<stem>_xyCorrected.tif`). Matching on a `C=\d+` token instead
  silently mispairs the `C3-` folders and swaps the two channels throughout Mode 2; the
  stem match is the fix, with the regex kept only as a fallback (section 3).

The folder names contain a non-ASCII `µ`. If discovery fails after moving the data between
platforms, rename to `30um/`, `60um/`, `90um/` and update `DATASETS` to match.

**Running.** Set `DATA_ROOT` in the config cell, then run every cell top to bottom.
All figures and tables are written to `outputs/<YYYYMMDD>_trial<N>/`. This notebook is
distributed without stored cell outputs; a full run regenerates all of them.


## 1. Setup

In [ ]:
%matplotlib inline
import os, glob
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.transforms import Bbox
from scipy.spatial import cKDTree
import warnings; warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110

# ---- Illustrator-editable vector export: keep text as real (non-outlined) glyphs ----
mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",   # keep text as text in SVG
    "font.family": "Arial",   # or another journal-approved font
})

def save_panel(fig, axes_group, path):
    # save a single panel (or an axes + its colorbar/legend) as its own cropped PDF asset
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    if not isinstance(axes_group, (list, tuple)):
        axes_group = [axes_group]
    bbox = Bbox.union([a.get_tightbbox(renderer) for a in axes_group])
    bbox = bbox.transformed(fig.dpi_scale_trans.inverted())
    fig.savefig(path, bbox_inches=bbox)

print('ready')


## 2. ✏️ Configuration — select files + set parameters

**`DATASETS`**: one entry per `size × replicate`. Each value can be:
- `None` — not available yet (skipped),
- a **folder path** (string) — auto-finds the `C=1` (Raji) and `C=2` (macrophage) `*spots*.csv`
  inside (preferring a `filtered` Raji file), **or**
- an explicit dict `{'raji': '<C=1 spots.csv>', 'mac': '<C=2 spots.csv>', 'bg': '<optional .tif>'}`.

**Parameters**: `RADIUS_FACTOR` (1.0 = B-cell centre within the macrophage disc; lower = stricter),
`MIN_FRAMES` (3 ≈ 45 min sustained contact), `GAP_TOL` (tolerated dropout inside an event),
`END_TOL` (frames a B cell may vanish after overlap and still count as ingested). `FACTOR_MAX` is
the loosest radius contacts are pre-computed at — the sweep and `retune()` can use any factor **≤
`FACTOR_MAX`** for free (raise it if you want to sweep looser).

In [ ]:
# ============================ EDIT: YOUR DATA FILES ============================
# DATA_ROOT = the folder that holds the per-condition folders listed below.
# Leave '' to use the folder this notebook is launched from, or set it to your own
# path (absolute, or relative to the notebook).
DATA_ROOT = ''                           # <- EDIT: e.g. '/path/to/PULP_phagocytosis_data'
BASE = os.path.abspath(os.path.expanduser(DATA_ROOT)) if DATA_ROOT else os.getcwd()
# Each entry is the folder for one acquisition; both channels (Raji C=1 + macrophage)
# are auto-discovered inside it and told apart by cell radius (see cell below),
# because channel numbering (C=0/1/2/3) is inconsistent across these datasets.
DATASETS = {
    '30um': {'R1': '30 µm/R1',
             'R3': '30 µm/R3',
             'R4': '30 µm/R4'},
    '60um': {'R2': '60 µm/R2',
             'R4': '60 µm/R4',
             'R5': '60 µm/R5'},
    '90um': {'R1': '90 µm/R1',
             'R2': '90 µm/R2',
             'R5': '90 µm/R5'},
}
# =================================================================================

# ========================= EDIT: DETECTION PARAMETERS =========================
PIXEL_UM      = 0.8    # micrometres per pixel (metadata)
MIN_PER_FRAME = 15     # minutes between frames
RADIUS_FACTOR = 1.25    # overlap threshold = this * macrophage radius
MIN_FRAMES    = 4      # sustained contact frames required for an event
GAP_TOL       = 1      # allow this many missing frames inside one event
END_TOL       = 3      # B cell may vanish this many frames after overlap and still count ingested
FACTOR_MAX    = 1.5    # contacts pre-computed at this loosest radius (sweep/retune must stay <= this)
RAJI_MAX_RADIUS_UM = 9.0  # spots with median radius below this = Raji (C=1); above = macrophage
# =================================================================================

SIZE_ORDER = ['30um', '60um', '90um']
SIZE_COLORS = {'30um': '#DAEFD1', '60um': '#F68F92', '90um': '#9CA9D2'}
def size_label(s):
    return s.replace('um', ' µm')   # display form for figures: '30um' -> '30 µm'
assert RADIUS_FACTOR <= FACTOR_MAX, 'RADIUS_FACTOR must be <= FACTOR_MAX'

# ============================ EDIT: OUTPUT FOLDER ============================
from datetime import datetime
TRIAL = None      # trial-number suffix (folder = outputs/<YYYYMMDD>_trial<TRIAL>); set None to auto-increment
_OUTBASE = os.path.join(BASE, 'outputs')
os.makedirs(_OUTBASE, exist_ok=True)
_TODAY = datetime.now().strftime('%Y%m%d')
if TRIAL is None:
    _pref = _TODAY + '_trial'
    _nums = [int(os.path.basename(p)[len(_pref):]) for p in glob.glob(os.path.join(_OUTBASE, _pref + '*'))
             if os.path.basename(p)[len(_pref):].isdigit()]
    TRIAL = (max(_nums) + 1) if _nums else 1
OUTDIR = os.path.join(_OUTBASE, f'{_TODAY}_trial{TRIAL}')
os.makedirs(OUTDIR, exist_ok=True)
def O(name):
    return os.path.join(OUTDIR, name)
print(f'Outputs -> {os.path.relpath(OUTDIR, BASE)}')   # relative, so no local path is stored in the notebook
# =============================================================================


## 3. File discovery & loading

`discover_spots` locates the two channels' spots CSVs in a folder (case-insensitive, matches either
`...C=1...spots...` or `...spots...C=1...`, prefers a `filtered` Raji export). `load_spots` reads a
TrackMate spots CSV, dropping the 3 alternate-name header rows and the units row, keeping only
spots that belong to a track.

In [ ]:
import re

def _median_radius(path):
    # cheap read of just ID + RADIUS to tell Raji (~6um) from macrophage (~13um)
    try:
        df = pd.read_csv(path, usecols=['ID', 'RADIUS'], low_memory=False)
        df = df[pd.to_numeric(df['ID'], errors='coerce').notna()]
        return float(pd.to_numeric(df['RADIUS'], errors='coerce').median())
    except Exception:
        return float('nan')


def discover_spots(folder):
    # classify every *spots*.csv by radius, NOT by channel number (which is inconsistent)
    spots = sorted(glob.glob(os.path.join(folder, '*spots*.csv')))
    raji_c, mac_c = [], []
    for p in spots:
        r = _median_radius(p)
        if np.isnan(r):
            continue
        (raji_c if r < RAJI_MAX_RADIUS_UM else mac_c).append((r, p))

    def pick(cands):
        if not cands:
            return None
        filt = [p for _, p in cands if 'filter' in os.path.basename(p).lower()]
        return filt[0] if filt else sorted(cands)[0][1]

    raji_p = pick(raji_c)
    mac_p = pick(mac_c)

    # Background movie = the macrophage channel's drift-corrected tif. Pair it to the
    # macrophage spots file on their shared stem: '<stem>_spots.csv' <-> '<stem>_xyCorrected.tif'.
    # (Matching on a 'C=\d+' token instead silently misses the folders named '..._C3-...',
    #  and the tifs[0] fallback then hands back the *Raji* movie -- which swapped the two
    #  channels throughout Mode 2 for 90um R1 and R2. Kept only as a secondary fallback.)
    bg = None
    tifs = sorted(glob.glob(os.path.join(folder, '*xyCorrected*.tif')))
    if mac_p and tifs:
        base = os.path.basename(mac_p)
        stem = base[:-len('_spots.csv')] if base.endswith('_spots.csv') else os.path.splitext(base)[0]
        match = [t for t in tifs if os.path.basename(t).startswith(stem)]
        if not match:
            m = re.search(r'C=\d+', base)
            if m:
                match = [t for t in tifs if m.group(0) in os.path.basename(t)]
        bg = match[0] if match else None
    if bg is None and tifs:
        bg = tifs[0]
    return raji_p, mac_p, bg


def resolve_entry(entry):
    if entry is None:
        return None
    if isinstance(entry, dict):
        return entry.get('raji'), entry.get('mac'), entry.get('bg')
    folder = entry if os.path.isabs(entry) else os.path.join(BASE, entry)
    return discover_spots(folder) if os.path.isdir(folder) else None


def load_spots(path):
    df = pd.read_csv(path, low_memory=False)
    df = df[pd.to_numeric(df['ID'], errors='coerce').notna()].copy()
    for c in ['TRACK_ID', 'FRAME', 'POSITION_X', 'POSITION_Y', 'RADIUS']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df = df.dropna(subset=['TRACK_ID', 'FRAME', 'POSITION_X', 'POSITION_Y'])
    df['FRAME'] = df['FRAME'].astype(int)
    df['TRACK_ID'] = df['TRACK_ID'].astype(int)
    return df



## 4. Core detection: co-localisation → events → ingestion flag

- `find_contacts` — for every frame, find each B cell within `FACTOR_MAX` × a macrophage's radius
  (fast per-frame `cKDTree`). Stores the **distance** and **macrophage radius** so any stricter
  factor can be applied later for free.
- `filter_contacts` — keep contacts with `dist ≤ factor × mac_radius` (this is the retuning knob).
- `build_events` — group filtered contacts by `(macrophage, B-cell)` into sustained runs (allowing
  `GAP_TOL` dropouts); runs ≥ `MIN_FRAMES` become events, each flagged `ingested`.
- `add_origins` — attach each ingested B cell's **origin** (first-frame position) and **ingestion
  time** (frame it vanishes → hours).

In [ ]:
def find_contacts(raji, mac, factor_max):
    rows = []
    frames = sorted(set(raji['FRAME']) & set(mac['FRAME']))
    for f in frames:
        rf = raji[raji['FRAME'] == f]; mf = mac[mac['FRAME'] == f]
        if len(rf) == 0 or len(mf) == 0:
            continue
        rpts = rf[['POSITION_X', 'POSITION_Y']].to_numpy()
        mpts = mf[['POSITION_X', 'POSITION_Y']].to_numpy()
        mrad = mf['RADIUS'].to_numpy()
        r_tids = rf['TRACK_ID'].to_numpy(); m_tids = mf['TRACK_ID'].to_numpy()
        tree = cKDTree(rpts)
        neigh = tree.query_ball_point(mpts, mrad * factor_max)
        for mi, ridxs in enumerate(neigh):
            for ri in ridxs:
                d = float(np.hypot(mpts[mi][0] - rpts[ri][0], mpts[mi][1] - rpts[ri][1]))
                rows.append((f, int(m_tids[mi]), int(r_tids[ri]), d, float(mrad[mi]),
                             (mpts[mi][0] + rpts[ri][0]) / 2, (mpts[mi][1] + rpts[ri][1]) / 2))
    return pd.DataFrame(rows, columns=['frame', 'mac_tid', 'raji_tid', 'dist', 'mac_r', 'x', 'y'])


def filter_contacts(contacts, factor):
    if len(contacts) == 0:
        return contacts
    return contacts[contacts['dist'] <= factor * contacts['mac_r']]


def build_events(contacts, raji, min_frames, gap_tol, end_tol):
    cols = ['mac_tid','raji_tid','start_frame','end_frame','n_contact_frames',
            'span_frames','duration_min','raji_last_frame','ingested','x_um','y_um']
    if len(contacts) == 0:
        return pd.DataFrame(columns=cols)
    raji_last = raji.groupby('TRACK_ID')['FRAME'].max().to_dict()
    last_frame = int(raji['FRAME'].max())
    events = []
    for (mtid, rtid), grp in contacts.groupby(['mac_tid', 'raji_tid']):
        grp = grp.sort_values('frame'); fr_all = grp['frame'].to_numpy()
        splits = np.where(np.diff(fr_all) > gap_tol + 1)[0] + 1
        for run in np.split(np.arange(len(fr_all)), splits):
            fr = fr_all[run]
            if len(fr) < min_frames:
                continue
            sub = grp.iloc[run]; rend = raji_last[rtid]
            ingested = (fr[0] <= rend <= fr[-1] + end_tol) and (rend < last_frame)
            events.append({
                'mac_tid': mtid, 'raji_tid': rtid,
                'start_frame': int(fr[0]), 'end_frame': int(fr[-1]),
                'n_contact_frames': len(fr), 'span_frames': int(fr[-1] - fr[0] + 1),
                'duration_min': int(fr[-1] - fr[0]) * MIN_PER_FRAME,
                'raji_last_frame': int(rend), 'ingested': bool(ingested),
                'x_um': round(sub['x'].mean(), 1), 'y_um': round(sub['y'].mean(), 1),
            })
    return pd.DataFrame(events, columns=cols).sort_values(['start_frame', 'mac_tid']).reset_index(drop=True)


def add_origins(events, raji):
    first = raji.sort_values('FRAME').groupby('TRACK_ID').first()
    events = events.copy()
    events['orig_x'] = events['raji_tid'].map(first['POSITION_X'])
    events['orig_y'] = events['raji_tid'].map(first['POSITION_Y'])
    events['ingest_hr'] = events['raji_last_frame'] * MIN_PER_FRAME / 60.0
    return events


def square_bounds(raji):
    xlo, xhi = np.percentile(raji['POSITION_X'], [0.5, 99.5])
    ylo, yhi = np.percentile(raji['POSITION_Y'], [0.5, 99.5])
    return xlo, xhi, ylo, yhi

## 5. Process every configured dataset

Loops over `DATASETS`, skips unavailable entries, and builds one **combined events table** tagged
with `size`/`replicate`. Contacts are computed **once per dataset at `FACTOR_MAX`** and cached in
`per_dataset`, so retuning and sweeps (§10) are instant.

In [ ]:
def process_all(radius_factor=None, min_frames=None, gap_tol=None, end_tol=None, verbose=True):
    # (re)build events for every dataset from cached contacts; caches contacts on first call
    rf = RADIUS_FACTOR if radius_factor is None else radius_factor
    mf = MIN_FRAMES if min_frames is None else min_frames
    gt = GAP_TOL if gap_tol is None else gap_tol
    et = END_TOL if end_tol is None else end_tol
    rows = []
    for size in SIZE_ORDER:
        for rep, entry in DATASETS.get(size, {}).items():
            key = (size, rep)
            if key not in per_dataset:
                res = resolve_entry(entry)
                if not res or not res[0] or not res[1]:
                    if verbose: print(f'skip  {size} {rep}: no spots files')
                    continue
                raji_p, mac_p, bg = res
                raji = load_spots(raji_p); mac = load_spots(mac_p)
                contacts = find_contacts(raji, mac, FACTOR_MAX)   # cached at loosest radius
                per_dataset[key] = dict(raji=raji, mac=mac, bg=bg, contacts=contacts, src=raji_p)
            d = per_dataset[key]
            ev = add_origins(build_events(filter_contacts(d['contacts'], rf), d['raji'], mf, gt, et), d['raji'])
            ev['size'] = size; ev['replicate'] = rep
            d['events'] = ev
            rows.append(ev)
            if verbose:
                n, ni = len(ev), int(ev['ingested'].sum())
                print(f'OK    {size} {rep}: {n} events, {ni} ingestion-confirmed '
                      f'({100*ni/max(n,1):.0f}%)  [{os.path.basename(d["src"])}]')
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


per_dataset = {}
combined = process_all()
print(f'\nProcessed {len(per_dataset)} dataset(s); {len(combined)} total events')

## 6. Per-dataset spatial map + where/when heatmaps

`plot_dataset` renders, for one dataset: the phagocytosis-event map (ingested events highlighted),
the **WHERE** heatmap (origin of ingested B cells within the square filter), and the **WHEN**
distribution (time of ingestion).

In [ ]:
def plot_dataset(size, rep, save=False):
    d = per_dataset.get((size, rep))
    if d is None:
        print(f'{size} {rep} not processed'); return
    ev = d['events']; raji = d['raji']
    ing = ev[ev['ingested']].dropna(subset=['orig_x', 'orig_y'])
    xlo, xhi, ylo, yhi = square_bounds(raji)

    fig, ax = plt.subplots(1, 3, figsize=(21, 6))
    no = ev[~ev['ingested']]; yes = ev[ev['ingested']]
    sc = ax[0].scatter(no.x_um, no.y_um, c=no.duration_min, cmap='viridis', s=22, alpha=0.7,
                       label='contact only')
    ax[0].scatter(yes.x_um, yes.y_um, marker='*', s=150, c='red', edgecolor='white',
                  linewidth=0.5, label='ingestion-confirmed')
    plt.colorbar(sc, ax=ax[0], fraction=0.046, pad=0.04).set_label('duration (min)')
    ax[0].legend(loc='upper right', fontsize=9)
    ax[0].set_title(f'{size_label(size)} {rep}: events (n={len(ev)}, {len(yes)} confirmed)', fontweight='bold')
    ax[0].set_xlabel('X (um)'); ax[0].set_ylabel('Y (um)')
    ax[0].set_xlim(xlo, xhi); ax[0].set_ylim(yhi, ylo)
    if len(ing):
        hb = ax[1].hist2d(ing.orig_x, ing.orig_y, bins=16, range=[[xlo, xhi], [ylo, yhi]], cmap='magma')
        plt.colorbar(hb[3], ax=ax[1], fraction=0.046, pad=0.04).set_label('# ingested B cells')
    ax[1].add_patch(Rectangle((xlo, ylo), xhi-xlo, yhi-ylo, fill=False, ec='cyan', lw=1.3, ls='--'))
    ax[1].set_title(f'{size_label(size)} {rep}: WHERE ingested B cells originated', fontweight='bold')
    ax[1].set_xlabel('X (um)'); ax[1].set_ylabel('Y (um)')
    ax[1].set_xlim(xlo, xhi); ax[1].set_ylim(yhi, ylo)
    ax[2].hist(ing.ingest_hr, bins=np.arange(0, 25, 1), color='#C0392B', edgecolor='black', alpha=0.85)
    ax[2].set_title(f'{size_label(size)} {rep}: WHEN ingestion occurs', fontweight='bold')
    ax[2].set_xlabel('time of ingestion (hours)'); ax[2].set_ylabel('# ingestion events')
    ax[2].set_xlim(0, 24); ax[2].set_xticks(np.arange(0, 25, 4)); ax[2].grid(alpha=0.3)
    plt.tight_layout()
    if save:
        plt.savefig(O(f'phago_{size}_{rep}.pdf'), bbox_inches='tight')
    plt.show()

for (size, rep) in per_dataset:
    plot_dataset(size, rep)


## 7. Summary table across conditions
One row per `size × replicate` → `phago_condition_summary.csv`.

In [ ]:
def make_summary(df):
    def summarise(g):
        n = len(g); ni = int(g['ingested'].sum())
        n_mac = g['mac_tid'].nunique()                          # NEW
        return pd.Series({
            'n_events': n, 'n_ingested': ni,
            'ingest_rate_%': round(100 * ni / max(n, 1), 1),
            'n_macrophages': n_mac, 'n_bcell_targets': g['raji_tid'].nunique(),
            'ingested_per_mac': round(ni / max(n_mac, 1), 3),    # NEW
            'events_per_mac': round(n / max(n_mac, 1), 3),       # NEW
            'median_ingest_hr': round(g.loc[g['ingested'], 'ingest_hr'].median(), 1) if ni else np.nan,
            'median_duration_min': round(g['duration_min'].median(), 0),
        })
    return df.groupby(['size', 'replicate']).apply(summarise).reset_index()

if len(combined):
    summary = make_summary(combined)
    summary.to_csv(O('phago_condition_summary.csv'), index=False)
    display(summary)
else:
    print('No datasets processed yet — fill in DATASETS (cell 2).')

In [ ]:
# ---- true total macrophage count per (size, replicate), from raw spots (not just contact-makers) ----
mac_totals = pd.DataFrame([
    {'size': size, 'replicate': rep, 'n_macrophages_total': d['mac']['TRACK_ID'].nunique()}
    for (size, rep), d in per_dataset.items()
])
display(mac_totals)

# ---- how many macrophages never made a qualifying contact? ----
summary = summary.merge(mac_totals, on=['size', 'replicate'], how='left')
summary['n_macrophages_idle'] = summary['n_macrophages_total'] - summary['n_macrophages']
summary['pct_macrophages_idle'] = (100 * summary['n_macrophages_idle']
                                    / summary['n_macrophages_total'].clip(lower=1)).round(1)
display(summary[['size', 'replicate', 'n_macrophages_total', 'n_macrophages',
                  'n_macrophages_idle', 'pct_macrophages_idle']])


## 8. Cross-condition comparison vs island size
**A** ingestion counts per replicate by size · **B** time-of-ingestion by size · **C** ingested
B-cell origin heatmap per size (replicates pooled). Renders gracefully with any subset.

In [ ]:
if len(combined):
    np.random.seed(0)
    sizes_present = [s for s in SIZE_ORDER if s in combined['size'].unique()]
    cnt = (combined.groupby(['size', 'replicate'])
           .agg(contacts=('ingested', 'size'), ingested=('ingested', 'sum'),
                n_macrophages=('mac_tid', 'nunique'))
           .reset_index())
    cnt['ingested_per_mac'] = cnt['ingested'] / cnt['n_macrophages'].clip(lower=1)
    cnt['contacts_per_mac'] = cnt['contacts'] / cnt['n_macrophages'].clip(lower=1)

    def violin_scatter(ax, values_by_size, ylabel, title):
        for i, s in enumerate(sizes_present):
            y = np.asarray(values_by_size[s], float)
            col = SIZE_COLORS.get(s, '#999')
            if len(y) >= 2 and np.ptp(y) > 0:
                parts = ax.violinplot([y], positions=[i], widths=0.7, showextrema=False)
                for pc in parts['bodies']:
                    pc.set_facecolor(col); pc.set_alpha(0.30)
                    pc.set_edgecolor(col); pc.set_linewidth(1.3)
            x = np.full(len(y), i) + np.random.uniform(-0.07, 0.07, len(y))
            ax.scatter(x, y, color=col, edgecolor='white', linewidth=0.6, s=75, zorder=3)
            ax.hlines(np.median(y), i - 0.22, i + 0.22, color='black', lw=2.2, zorder=4)
        ax.set_xticks(range(len(sizes_present))); ax.set_xticklabels([size_label(x) for x in sizes_present])
        ax.set_ylabel(ylabel); ax.set_title(title, fontweight='bold')
        ax.grid(alpha=0.3, axis='y'); ax.set_ylim(bottom=0)

    ing_by = {s: cnt[cnt['size'] == s]['ingested'].values for s in sizes_present}
    con_by = {s: cnt[cnt['size'] == s]['contacts'].values for s in sizes_present}
    fig, ax = plt.subplots(1, 2, figsize=(13, 5.2))
    violin_scatter(ax[0], ing_by, 'ingestion-confirmed events / replicate',
                   'A) Phagocytosis (ingestion) vs island size')
    violin_scatter(ax[1], con_by, 'contact events / replicate',
                   'B) Contacts vs island size')
    fig.text(0.5, -0.02, 'violin = distribution across replicates; dot = one replicate; black bar = median',
             ha='center', fontsize=9, color='#555')
    plt.tight_layout()
    plt.savefig(O('phago_vs_size.pdf'), bbox_inches='tight')
    save_panel(fig, ax[0], O('phago_vs_size_A.pdf'))
    save_panel(fig, ax[1], O('phago_vs_size_B.pdf'))
    plt.show()

    # --- same comparison, normalized per macrophage (controls for cell count differing by island size) ---
    ing_pm_by = {s: cnt[cnt['size'] == s]['ingested_per_mac'].values for s in sizes_present}
    con_pm_by = {s: cnt[cnt['size'] == s]['contacts_per_mac'].values for s in sizes_present}
    fig, ax = plt.subplots(1, 2, figsize=(13, 5.2))
    violin_scatter(ax[0], ing_pm_by, 'ingestion-confirmed events / macrophage',
                   'C) Phagocytosis per macrophage vs island size')
    violin_scatter(ax[1], con_pm_by, 'contact events / macrophage',
                   'D) Contacts per macrophage vs island size')
    fig.text(0.5, -0.02, 'normalized by n_macrophages in that replicate; violin = distribution across '
             'replicates; dot = one replicate; black bar = median', ha='center', fontsize=9, color='#555')
    plt.tight_layout()
    plt.savefig(O('phago_vs_size_per_macrophage.pdf'), bbox_inches='tight')
    save_panel(fig, ax[0], O('phago_vs_size_per_macrophage_C.pdf'))
    save_panel(fig, ax[1], O('phago_vs_size_per_macrophage_D.pdf'))
    plt.show()

    # --- timing: when ingestion occurs, by size ---
    fig, ax = plt.subplots(figsize=(8, 5))
    for s in sizes_present:
        hr = combined[(combined['size'] == s) & (combined['ingested'])]['ingest_hr'].dropna()
        if len(hr):
            ax.hist(hr, bins=np.arange(0, 25, 1), histtype='step', linewidth=2.2,
                    color=SIZE_COLORS.get(s, '#999'), label=size_label(s), density=True)
    ax.set_xlabel('time of ingestion (hours)'); ax.set_ylabel('density')
    ax.set_title('When ingestion occurs, by size', fontweight='bold')
    ax.legend(title='island size'); ax.set_xlim(0, 24); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(O('phago_when_by_size.pdf'), bbox_inches='tight')
    plt.show()

    # --- where ingested B cells originated, by size ---
    fig, axes = plt.subplots(1, len(sizes_present), figsize=(7*len(sizes_present), 6), squeeze=False)
    for j, s in enumerate(sizes_present):
        ax = axes[0][j]
        rajis = [per_dataset[k]['raji'] for k in per_dataset if k[0] == s]
        xb = np.concatenate([r['POSITION_X'].values for r in rajis])
        yb = np.concatenate([r['POSITION_Y'].values for r in rajis])
        xlo, xhi = np.percentile(xb, [0.5, 99.5]); ylo, yhi = np.percentile(yb, [0.5, 99.5])
        ing = combined[(combined['size'] == s) & (combined['ingested'])].dropna(subset=['orig_x', 'orig_y'])
        if len(ing):
            hb = ax.hist2d(ing.orig_x, ing.orig_y, bins=16, range=[[xlo, xhi], [ylo, yhi]], cmap='magma')
            plt.colorbar(hb[3], ax=ax, fraction=0.046, pad=0.04).set_label('# ingested B cells')
        ax.set_title(f'{size_label(s)}: ingested B-cell origins (n={len(ing)})', fontweight='bold')
        ax.set_xlabel('X (um)'); ax.set_ylabel('Y (um)'); ax.set_xlim(xlo, xhi); ax.set_ylim(yhi, ylo)
    plt.tight_layout()
    plt.savefig(O('phago_where_by_size.pdf'), bbox_inches='tight')
    plt.show()
else:
    print('No datasets processed yet - fill in DATASETS (cell 2).')


## 8b. WHERE contacts & ingestion happen, in 8-hour phases

For each island size (rows) and 8-hour window (**Early 0-8 h / Mid 8-16 h / Late 16-24 h**,
binned by when the contact starts): grey dots = every macrophage-Raji contact, red dots =
the contacts that end in a confirmed ingestion (red = Raji, per the cell-type convention).
Dashed line = island bounds.

In [ ]:
# ============ WHERE contacts & ingestion happen, in 8-hour phases ============
PHASES = [('Early (0-8 h)', 0, 8), ('Mid (8-16 h)', 8, 16), ('Late (16-24 h)', 16, 24)]

if len(combined):
    sizes_present = [s for s in SIZE_ORDER if s in combined['size'].unique()]
    ev_all = combined.dropna(subset=['x_um', 'y_um']).copy()
    ev_all['start_hr'] = ev_all['start_frame'] * MIN_PER_FRAME / 60.0
    nrow, ncol = len(sizes_present), len(PHASES)
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.6*ncol, 4.6*nrow), squeeze=False)
    for i, s in enumerate(sizes_present):
        rajis = [per_dataset[k]['raji'] for k in per_dataset if k[0] == s]
        xb = np.concatenate([r['POSITION_X'].values for r in rajis])
        yb = np.concatenate([r['POSITION_Y'].values for r in rajis])
        xlo, xhi = np.percentile(xb, [0.5, 99.5]); ylo, yhi = np.percentile(yb, [0.5, 99.5])
        s_ev = ev_all[ev_all['size'] == s]
        for j, (plabel, t0, t1) in enumerate(PHASES):
            ax = axes[i][j]
            sub = s_ev[(s_ev['start_hr'] >= t0) & (s_ev['start_hr'] < t1)]
            con = sub[~sub['ingested']]; ing = sub[sub['ingested']]
            ax.scatter(con.x_um, con.y_um, s=13, c='#9aa0a6', alpha=0.28, edgecolor='none')
            ax.scatter(ing.x_um, ing.y_um, s=44, c='#E23B3B', alpha=0.75,
                       edgecolor='white', linewidth=0.4)
            ax.add_patch(Rectangle((xlo, ylo), xhi-xlo, yhi-ylo, fill=False, ec='cyan', lw=1.2, ls='--'))
            ax.set_xlim(xlo, xhi); ax.set_ylim(yhi, ylo)
            ax.set_xticks([]); ax.set_yticks([])
            ax.text(0.04, 0.965, f'contacts={len(con)}\ning={len(ing)}', transform=ax.transAxes,
                    va='top', ha='left', fontsize=9, color='#222', fontweight='bold')
            if i == 0: ax.set_title(plabel, fontweight='bold', fontsize=12)
            if j == 0: ax.set_ylabel(size_label(s), fontweight='bold', fontsize=13, rotation=0,
                                     ha='right', va='center', labelpad=28)
    handles = [plt.Line2D([], [], marker='o', color='w', markerfacecolor='#9aa0a6', markersize=8,
                          label='contact only'),
               plt.Line2D([], [], marker='o', color='w', markerfacecolor='#E23B3B', markersize=9,
                          label='ingestion-confirmed')]
    fig.legend(handles=handles, loc='lower center', ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.015), fontsize=11)
    fig.suptitle('WHERE contacts (grey) & ingestion (red) occur, by 8-hour phase (binned by contact onset)',
                 fontweight='bold', y=1.012)
    plt.tight_layout()
    plt.savefig(O('phago_where_by_phase.pdf'), bbox_inches='tight'); plt.show()
else:
    print('No datasets processed yet - fill in DATASETS (cell 2).')


## 8b-ii. Per-replicate WHERE maps, over the micropatterned islands

Section 8b pools the three replicates of each island size into one row. Here **each
acquisition is drawn on its own**, over that replicate's island mask, so a single field can
be read against the pattern rather than against an empty background — grey = every contact,
red = the contacts that end in a confirmed ingestion, binned into the same 8-hour phases by
when the contact starts.

The mask is a hand-drawn binary TIFF on the movie's own pixel grid, and event positions are
TrackMate microns; both are placed in one micron coordinate system via `PIXEL_UM`, with the
axes spanning the true image extent (`W × PIXEL_UM` by `H × PIXEL_UM`), so the overlay is
exact rather than fitted. A mask whose pixel dimensions do not match that replicate's movie
is refused rather than drawn (`allow_shape_mismatch=True` overrides).

Only Mode 1 is involved — no colocalization — so this section is independent of 8c.
Options: `zones_um=` shades the island rim separately from its interior (rim band measured
inwards from the boundary; islands cut by the field of view are drawn flat and kept out of
the split, since their real rim lies outside the image), and `show_on_island=True` annotates
what fraction of contacts and ingestions land on a pattern.


In [ ]:
# ============ WHERE by phase, PER REPLICATE, over the micropatterned islands ============
import tifffile
from matplotlib.colors import to_rgba
from scipy import ndimage

# ---- island masks -------------------------------------------------------------------
# Each replicate folder carries a hand-made binary mask of the micropatterned islands,
# drawn on the same pixel grid as that replicate's movie stack, so mask pixel (row, col)
# is movie pixel (row, col) and no rescaling is involved.
MASK_GLOB = 'BCellM0_*mask_whole.tif'
EDGE_UM   = 5.0        # rim band width, measured inwards from the island boundary
ZONE_OFF, ZONE_EDGE, ZONE_CORE, ZONE_CLIPPED = 0, 1, 2, 3

CONTACT_COL, CONTACT_ALPHA, CONTACT_SIZE = '#4a5058', 0.45, 15   # darker than 8b: the
INGEST_COL = '#E23B3B'                                           # island fill sits behind
ISLAND_FILL, ISLAND_EDGE = '#d7e6f7', '#7f9dc0'
RIM_FILL, CORE_FILL = '#9dc0e8', '#e8f1fb'


def pixel_extent(H, W):
    """imshow extent in microns for an HxW pixel grid, aligned to pixel centres.

    TrackMate reports a position as (pixel index x calibration), so micron 0.0 is the
    CENTRE of pixel 0; imshow's extent gives outer pixel boundaries, hence the half-pixel
    offset. It is only 0.4 um, but it keeps the drawn mask and `on_island()` (which rounds
    microns straight to an index) referring to exactly the same pixels.
    """
    half = 0.5 * PIXEL_UM
    return (-half, (W - 0.5) * PIXEL_UM, (H - 0.5) * PIXEL_UM, -half)


def find_mask(size, rep):
    entry = DATASETS.get(size, {}).get(rep)
    if not isinstance(entry, str):
        return None
    folder = entry if os.path.isabs(entry) else os.path.join(BASE, entry)
    hits = sorted(glob.glob(os.path.join(folder, MASK_GLOB)))
    return hits[0] if hits else None


def load_mask(size, rep, verbose=True):
    """(mask, extent) for one replicate, or (None, None) when the folder has no mask."""
    path = find_mask(size, rep)
    if path is None:
        if verbose:
            print(f'  ! {size} {rep}: no {MASK_GLOB} in the folder -- drawn without islands')
        return None, None
    m = tifffile.imread(path)
    if m.ndim > 2:                      # a stack would mean the mask is not a flat ROI
        m = m[0]
    m = m > 0
    return m, pixel_extent(*m.shape)


def movie_shape(size, rep):
    """(H, W) of the replicate's macrophage movie, or None."""
    bg = per_dataset.get((size, rep), {}).get('bg')
    if bg is None:
        return None
    with tifffile.TiffFile(bg) as t:
        return tuple(t.pages[0].shape[-2:])


def on_island(mask, x_um, y_um):
    """Boolean: does each (x, y) micron position fall on a patterned island?"""
    if mask is None:
        return None
    H, W = mask.shape
    xi = np.clip(np.round(np.asarray(x_um, float) / PIXEL_UM).astype(int), 0, W - 1)
    yi = np.clip(np.round(np.asarray(y_um, float) / PIXEL_UM).astype(int), 0, H - 1)
    return mask[yi, xi]


def island_zones(mask, edge_um=EDGE_UM):
    """Split an island mask into rim band vs interior -> (zones, areas).

    Islands cut off by the edge of the field are labelled ZONE_CLIPPED and kept out of the
    edge/core split entirely: their real rim lies outside the image, so the distance
    transform would call the invisible part "core" and bias the result.
    """
    mask = np.asarray(mask, bool)
    lab, _ = ndimage.label(mask)
    touching = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]])))
    touching.discard(0)
    clipped = np.isin(lab, list(touching)) & mask
    interior = mask & ~clipped

    dist_um = ndimage.distance_transform_edt(mask) * PIXEL_UM   # to nearest off-island px
    zones = np.full(mask.shape, ZONE_OFF, np.uint8)
    zones[clipped] = ZONE_CLIPPED
    zones[interior & (dist_um <= edge_um)] = ZONE_EDGE
    zones[interior & (dist_um > edge_um)] = ZONE_CORE
    areas = {'off': int((zones == ZONE_OFF).sum()), 'edge': int((zones == ZONE_EDGE).sum()),
             'core': int((zones == ZONE_CORE).sum()), 'clipped': int(clipped.sum())}
    return zones, areas


# ---- drawing ------------------------------------------------------------------------

def _island_layer(ax, mask, extent, fill=ISLAND_FILL, edge=ISLAND_EDGE, edge_px=2):
    """Island mask as a filled layer with its own outline, behind everything.

    Fill and outline go into ONE rgba array drawn by a single imshow, so they cannot drift
    apart. (Drawing the outline with `contour` instead is a trap: contour ignores `extent`
    unless `origin` is also passed, and silently falls back to array-index coordinates.)
    """
    rgba = np.zeros(mask.shape + (4,), float)
    rgba[mask] = to_rgba(fill)
    if edge_px:
        b = np.zeros_like(mask)
        for ax_ in (0, 1):
            for sh in (1, -1):
                b |= mask ^ np.roll(mask, sh, axis=ax_)
        if edge_px > 1:                  # thicken so the outline survives downscaling
            b = ndimage.binary_dilation(b, iterations=int(edge_px) - 1)
        rgba[b & mask] = to_rgba(edge)
    ax.imshow(rgba, extent=extent, interpolation='nearest', zorder=0)


def _zone_layer(ax, zones, extent):
    """Island fill split into rim band vs interior (island_zones codes)."""
    rgba = np.zeros(zones.shape + (4,), float)
    rgba[zones == ZONE_CORE] = to_rgba(CORE_FILL)
    rgba[zones == ZONE_EDGE] = to_rgba(RIM_FILL)
    # islands cut by the field of view can't be split; draw them flat so they read as
    # "not scored" rather than pretending to be all core
    rgba[zones == ZONE_CLIPPED] = to_rgba('#eeeeee')
    ax.imshow(rgba, extent=extent, interpolation='nearest', zorder=0)


# (named apart from the montage cell's own _scale_bar, which takes an image shape
# rather than a micron extent -- same idea, different units)
def _scale_bar_um(ax, extent, um=100, color='#222', frac_h=0.012, pad_frac=0.04):
    """Horizontal scale bar, bottom-right, in micron data coordinates."""
    x0, x1, y1, y0 = extent
    w, h = x1 - x0, y1 - y0
    bh = frac_h * h
    ax.add_patch(Rectangle((x1 - pad_frac * w - um, y1 - pad_frac * h - bh),
                           um, bh, facecolor=color, edgecolor='none', zorder=6))


def where_by_phase_replicate(size, rep, use_mask=True, bar_um=100, zones_um=None,
                             show_on_island=False, allow_shape_mismatch=False,
                             save=True, verbose=True):
    """The 8b map for ONE replicate, drawn over that replicate's island mask."""
    d = per_dataset.get((size, rep))
    if d is None or 'events' not in d:
        print(f'{size} {rep} not processed'); return
    ev = d['events'].dropna(subset=['x_um', 'y_um']).copy()
    ev['start_hr'] = ev['start_frame'] * MIN_PER_FRAME / 60.0

    mask, extent = (None, None)
    if use_mask:
        mask, extent = load_mask(size, rep, verbose=verbose)
        shape = movie_shape(size, rep) if mask is not None else None
        if shape is not None and tuple(shape) != tuple(mask.shape):
            msg = (f'{size} {rep}: mask is {mask.shape} but the movie is {shape}; '
                   f'microns would not line up')
            if not allow_shape_mismatch:
                raise ValueError(msg + ' -- refusing to draw. '
                                 'Pass allow_shape_mismatch=True to override.')
            print('  ! ' + msg + ' -- drawing anyway')
    if extent is None:
        shape = movie_shape(size, rep)      # no mask: use the movie's own extent, so the
        extent = pixel_extent(*shape)       # axes still mean microns

    isl = on_island(mask, ev['x_um'].to_numpy(), ev['y_um'].to_numpy())
    area_frac = float(mask.mean()) if mask is not None else None
    zmap = island_zones(mask, edge_um=zones_um)[0] if (mask is not None and zones_um) else None

    fig, axes = plt.subplots(1, len(PHASES), figsize=(4.9 * len(PHASES), 4.5), squeeze=False)
    for j, (plabel, t0, t1) in enumerate(PHASES):
        ax = axes[0][j]
        sel = ((ev['start_hr'] >= t0) & (ev['start_hr'] < t1)).to_numpy()
        sub = ev[sel]
        sub_isl = isl[sel] if isl is not None else None
        ing_m = sub['ingested'].to_numpy().astype(bool)
        con, ing = sub[~ing_m], sub[ing_m]

        if zmap is not None:
            _zone_layer(ax, zmap, extent)
        elif mask is not None:
            _island_layer(ax, mask, extent)
        ax.scatter(con.x_um, con.y_um, s=CONTACT_SIZE, c=CONTACT_COL, alpha=CONTACT_ALPHA,
                   edgecolor='none', zorder=2)
        ax.scatter(ing.x_um, ing.y_um, s=44, c=INGEST_COL, alpha=0.75,
                   edgecolor='white', linewidth=0.4, zorder=3)
        _scale_bar_um(ax, extent, um=bar_um)

        ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3])
        ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_color('#bbb')

        lines = [f'contacts = {len(con)}', f'ingestions = {len(ing)}']
        if show_on_island and sub_isl is not None and len(sub):
            pc = 100 * sub_isl[~ing_m].mean() if (~ing_m).any() else float('nan')
            pi = 100 * sub_isl[ing_m].mean() if ing_m.any() else float('nan')
            lines = [f'contacts = {len(con)}   ({pc:.0f}% on island)',
                     f'ingestions = {len(ing)}   ({pi:.0f}% on island)']
        # counts go under the panel, so they never sit on top of the data
        ax.set_xlabel('\n'.join(lines), fontsize=9.5, color='#222', fontweight='bold',
                      labelpad=6, linespacing=1.5)
        ax.set_title(plabel, fontweight='bold', fontsize=12)

    handles = [plt.Line2D([], [], marker='o', color='w', markerfacecolor=CONTACT_COL,
                          markersize=8, label='contact only'),
               plt.Line2D([], [], marker='o', color='w', markerfacecolor=INGEST_COL,
                          markersize=9, label='ingestion-confirmed')]
    if zmap is not None:
        handles += [plt.Line2D([], [], marker='s', color='w', markerfacecolor=RIM_FILL,
                               markeredgecolor='none', markersize=10,
                               label=f'island rim (\u2264{zones_um:g} µm)'),
                    plt.Line2D([], [], marker='s', color='w', markerfacecolor=CORE_FILL,
                               markeredgecolor=ISLAND_EDGE, markersize=10,
                               label='island interior')]
    elif mask is not None:
        handles.append(plt.Line2D([], [], marker='s', color='w', markerfacecolor=ISLAND_FILL,
                                  markeredgecolor=ISLAND_EDGE, markersize=10,
                                  label='patterned island'))
    # anchored clear of the per-panel count labels that sit under each axes
    fig.legend(handles=handles, loc='upper center', ncol=len(handles), frameon=False,
               bbox_to_anchor=(0.5, 0.02), fontsize=11)

    sub_t = f'{size_label(size)} {rep}: WHERE contacts (grey) & ingestion (red) occur, by 8-hour phase'
    if area_frac is not None and show_on_island:
        tot_c = ~ev['ingested'].to_numpy().astype(bool)
        tot_i = ev['ingested'].to_numpy().astype(bool)
        pc = 100 * isl[tot_c].mean() if tot_c.any() else float('nan')
        pi = 100 * isl[tot_i].mean() if tot_i.any() else float('nan')
        sub_t += (f'\nislands cover {100*area_frac:.0f}% of the field; overall '
                  f'{pc:.0f}% of contacts and {pi:.0f}% of ingestions fall on one'
                  f'   (scale bar {bar_um:g} µm)')
    else:
        sub_t += f'   (scale bar {bar_um:g} µm)'
    fig.suptitle(sub_t, fontweight='bold', y=1.02, fontsize=12)

    plt.tight_layout(rect=(0, 0.075, 1, 1))   # leave a strip at the bottom for the legend
    if save:
        out = O(f'phago_where_by_phase_{size}_{rep}.pdf')
        plt.savefig(out, bbox_inches='tight')
        if verbose:
            print(f'  wrote {os.path.basename(out)}')
    plt.show()


# one figure per replicate; `PHASES` is defined in 8b above
if len(combined):
    for _s in SIZE_ORDER:
        for _r in DATASETS.get(_s, {}):
            if (_s, _r) in per_dataset:
                where_by_phase_replicate(_s, _r)
else:
    print('No datasets processed yet - fill in DATASETS (cell 2).')


## 8c. Mode 2 — Signal colocalization (independent of the tracks)

The calls above come purely from tracked centroids. Here we cross-check *every* candidate
contact against the raw pixels. For each contact frame, at the macrophage centroid we
sample an ROI (`COLOC_PAD` x the macrophage radius) in both drift-corrected channels,
build the macrophage-body mask by Otsu-thresholding the blue channel, and compute
**Manders' M1** = the fraction of local Raji (red) signal inside that mask. A frame is
colocalized when M1 >= `COLOC_M1_MIN`; an event is **colocalization-confirmed** when this
persists for >= `MIN_FRAMES` frames with gap tolerance `GAP_TOL` — the same sustained-frame
limits as the tracking analysis. The plot below validates the tracking-called ingestions;
the full per-event result feeds the corroboration in 8d.

In [ ]:
# ============ Mode 2: signal colocalization over ALL candidate contacts ============
import tifffile
from skimage.filters import threshold_otsu

COLOC_M1_MIN = 0.5     # frame colocalized if >= this fraction of local red signal is inside the macrophage
COLOC_PAD    = 1.5     # ROI half-width = COLOC_PAD * macrophage radius (px)

def _find_corrected_tifs(size, rep):
    # macrophage movie = radius-resolved bg tif; Raji movie = the other xyCorrected tif
    entry = DATASETS.get(size, {}).get(rep)
    if not isinstance(entry, str):
        return None, None
    folder = entry if os.path.isabs(entry) else os.path.join(BASE, entry)
    tifs = sorted(f for f in glob.glob(os.path.join(folder, '*xyCorrected*.tif'))
                  if 'copy' not in os.path.basename(f).lower())
    mtif = per_dataset.get((size, rep), {}).get('bg')
    if mtif not in tifs:
        mtif = tifs[-1] if tifs else None
    rtif = next((t for t in tifs if t != mtif), None)
    return rtif, mtif

def _otsu_safe(a):
    a = a.astype(float)
    if a.max() <= a.min():
        return a.max() + 1
    try:
        return threshold_otsu(a)
    except Exception:
        return (a.min() + a.max()) / 2

def _stretch(a):
    a = a.astype(float)
    lo, hi = np.percentile(a, [2, 99])
    return np.clip((a - lo) / (hi - lo + 1e-8), 0, 1)

def _composite(R, B):
    comp = np.zeros((B.shape[-2], B.shape[-1], 3))
    comp[..., 0] = _stretch(R)     # red   = Raji
    comp[..., 2] = _stretch(B)     # blue  = macrophage
    return comp

def _manders_m1(red, blue, cx, cy, r_px, pad=None):
    padf = COLOC_PAD if pad is None else pad
    H, W = blue.shape[-2], blue.shape[-1]
    rad = int(max(6, r_px * padf))
    x0, x1 = max(0, int(cx-rad)), min(W, int(cx+rad))
    y0, y1 = max(0, int(cy-rad)), min(H, int(cy+rad))
    if x1 <= x0 or y1 <= y0:
        return np.nan
    r = red[y0:y1, x0:x1].astype(float); b = blue[y0:y1, x0:x1].astype(float)
    mask = b > _otsu_safe(b)
    r = np.clip(r - np.percentile(r, 20), 0, None)
    denom = r.sum()
    return 0.0 if denom <= 0 else float((r * mask).sum() / denom)

def _max_sustained(frames, min_frames, gap_tol):
    fr = np.array(sorted(set(frames)))
    if len(fr) == 0:
        return 0
    splits = np.where(np.diff(fr) > gap_tol + 1)[0] + 1
    return max((len(run) for run in np.split(fr, splits)), default=0)

def colocalize_dataset(size, rep, min_frames=None, gap_tol=None):
    mf = MIN_FRAMES if min_frames is None else min_frames
    gt = GAP_TOL if gap_tol is None else gap_tol
    d = per_dataset.get((size, rep))
    if d is None or 'events' not in d or len(d['events']) == 0:
        return pd.DataFrame()
    rtif, mtif = _find_corrected_tifs(size, rep)
    if rtif is None or mtif is None:
        print(f'  {size} {rep}: tif(s) missing - skipped'); return pd.DataFrame()
    mac = d['mac']
    lut = {}
    for tid, fr, x, y, r in zip(mac['TRACK_ID'], mac['FRAME'], mac['POSITION_X'],
                                mac['POSITION_Y'], mac['RADIUS']):
        lut[(int(tid), int(fr))] = (x/PIXEL_UM, y/PIXEL_UM, (r if pd.notna(r) else 12)/PIXEL_UM)
    rfh = tifffile.TiffFile(rtif); bfh = tifffile.TiffFile(mtif)
    nred, nblue = len(rfh.pages), len(bfh.pages)
    fcache = {}
    def gf(tf, key, i, n):
        if i < 0 or i >= n:
            return None
        k = (key, i)
        if k not in fcache:
            fcache[k] = tf.pages[i].asarray()
        return fcache[k]
    rows = []
    for e in d['events'].itertuples():
        pos = []
        for fn in range(int(e.start_frame), int(e.end_frame) + 1):
            p = lut.get((int(e.mac_tid), fn))
            if p is None:
                continue
            R = gf(rfh, 'r', fn, nred); B = gf(bfh, 'b', fn, nblue)
            if R is None or B is None:
                continue
            v = _manders_m1(R, B, p[0], p[1], p[2])
            if not np.isnan(v) and v >= COLOC_M1_MIN:
                pos.append(fn)
        run = _max_sustained(pos, mf, gt)
        rows.append({'size': size, 'replicate': rep, 'mac_tid': int(e.mac_tid),
                     'raji_tid': int(e.raji_tid), 'start_frame': int(e.start_frame),
                     'n_coloc_frames': len(pos), 'max_coloc_run': run,
                     'coloc_confirmed': bool(run >= mf)})
    rfh.close(); bfh.close()
    return pd.DataFrame(rows)

if len(combined):
    parts = []
    for (size, rep) in per_dataset:
        print(f'colocalizing {size} {rep} ...')
        parts.append(colocalize_dataset(size, rep))
    coloc_all = pd.concat([p for p in parts if len(p)], ignore_index=True) if any(len(p) for p in parts) else pd.DataFrame()

    KEY = ['size', 'replicate', 'mac_tid', 'raji_tid', 'start_frame']
    CF = KEY + ['n_coloc_frames', 'max_coloc_run', 'coloc_confirmed']
    combined = combined.drop(columns=[c for c in CF if c not in KEY and c in combined.columns], errors='ignore')
    combined = combined.merge(coloc_all[CF], on=KEY, how='left') if len(coloc_all) else combined.assign(
        n_coloc_frames=0, max_coloc_run=0, coloc_confirmed=False)
    combined['coloc_confirmed'] = combined['coloc_confirmed'].fillna(False)
    combined['max_coloc_run'] = combined['max_coloc_run'].fillna(0)
    for k in per_dataset:
        ev = per_dataset[k]['events']
        ev = ev.drop(columns=[c for c in CF if c not in KEY and c in ev.columns], errors='ignore')
        if len(coloc_all):
            ev = ev.merge(coloc_all[CF], on=KEY, how='left')
            ev['coloc_confirmed'] = ev['coloc_confirmed'].fillna(False)
            ev['max_coloc_run'] = ev['max_coloc_run'].fillna(0)
        per_dataset[k]['events'] = ev

    val = combined[combined['ingested']]
    summ = (val.groupby(['size', 'replicate'])
            .agg(n_ingest=('coloc_confirmed', 'size'), n_coloc=('coloc_confirmed', 'sum')).reset_index())
    summ['coloc_rate_%'] = (100 * summ['n_coloc'] / summ['n_ingest']).round(1)
    print()
    print(f'Tracking-called ingestions confirmed by colocalization: '
          f'{int(val["coloc_confirmed"].sum())}/{len(val)} ({100*val["coloc_confirmed"].mean():.0f}%)')
    display(summ)
else:
    print('No datasets processed yet - fill in DATASETS (cell 2).')


In [ ]:
# ---- colocalization validation of the tracking-called ingestions ----
if len(combined) and 'coloc_confirmed' in combined.columns:
    np.random.seed(1)
    sp = [s for s in SIZE_ORDER if s in summ['size'].unique()]
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    for i, s in enumerate(sp):
        y = summ[summ['size'] == s]['coloc_rate_%'].values
        col = SIZE_COLORS.get(s, '#999')
        if len(y) >= 2 and np.ptp(y) > 0:
            for pc in ax[0].violinplot([y], positions=[i], widths=0.7, showextrema=False)['bodies']:
                pc.set_facecolor(col); pc.set_alpha(0.30); pc.set_edgecolor(col); pc.set_linewidth(1.3)
        ax[0].scatter(np.full(len(y), i) + np.random.uniform(-0.07, 0.07, len(y)), y,
                      color=col, edgecolor='white', linewidth=0.6, s=75, zorder=3)
        ax[0].hlines(np.median(y), i-0.22, i+0.22, color='black', lw=2.2, zorder=4)
    ax[0].set_xticks(range(len(sp))); ax[0].set_xticklabels([size_label(x) for x in sp]); ax[0].set_ylim(0, 100)
    ax[0].set_ylabel('% of tracking-ingestion events\nconfirmed by colocalization')
    ax[0].set_title('A) Signal-colocalization confirmation rate', fontweight='bold')
    ax[0].grid(alpha=0.3, axis='y')

    runs = val['max_coloc_run']
    hi = int(runs.max()) if len(runs) else MIN_FRAMES
    ax[1].hist(runs, bins=np.arange(0, hi+2)-0.5, color='#6a51a3', alpha=0.85)
    ax[1].axvline(MIN_FRAMES-0.5, color='red', ls='--', lw=2, label=f'MIN_FRAMES = {MIN_FRAMES}')
    ax[1].set_xlabel('max sustained colocalized frames per event')
    ax[1].set_ylabel('# tracking-ingestion events')
    ax[1].set_title('B) Sustained colocalization vs tracking threshold', fontweight='bold')
    ax[1].legend(); ax[1].grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(O('phago_colocalization_validation.pdf'), bbox_inches='tight'); plt.show()


## 8d. Corroboration — do the two modes agree? (Venn)

The two analyses each yield a *set* of events: **T = tracking-called ingestions** (the Raji
track vanished during contact) and **C = colocalization-confirmed contacts** (the Raji signal
stayed inside the macrophage for >= `MIN_FRAMES`). Their intersection **T & C** is the
*double-corroborated* set — the highest-confidence phagocytosis calls. The Venn diagrams below
(per island size and pooled) show |T only|, |T & C|, |C only| directly, and every headline
number is just a ratio of these regions:

- **double / T** = fraction of the tracking-called ingestions the pixels confirm (the "% of T");
- **double / C** = fraction of colocalized contacts the tracking also called an ingestion;
- events in **neither** set are plain contacts.

The double-corroborated events are exported (`phago_double_corroborated_events.csv`) and then
drawn on the movie image, coloured by class.

In [ ]:
# ============ corroboration as sets (Venn), explicit table, filter, show on image ============
CAT_ORDER = ['contact', 'tracking-only', 'coloc-only', 'double-corroborated']
CAT_COL = {'contact': '#9aa0a6', 'tracking-only': '#f39c12',
           'coloc-only': '#1abc9c', 'double-corroborated': '#2ecc71'}

def categorize(df):
    d = df.copy()
    ing = d['ingested'].fillna(False).astype(bool).to_numpy()
    col = (d['coloc_confirmed'].fillna(False).astype(bool).to_numpy()
           if 'coloc_confirmed' in d else np.zeros(len(d), bool))
    d['category'] = np.select([ing & col, ing & ~col, ~ing & col],
                              ['double-corroborated', 'tracking-only', 'coloc-only'], 'contact')
    return d

def _venn_counts(sub):
    ing = sub['ingested'].fillna(False).astype(bool)
    col = sub['coloc_confirmed'].fillna(False).astype(bool)
    return dict(t_only=int((ing & ~col).sum()), both=int((ing & col).sum()),
                c_only=int((~ing & col).sum()), neither=int((~ing & ~col).sum()),
                T=int(ing.sum()), C=int(col.sum()))

def _draw_venn(ax, c, title):
    r, dx = 0.62, 0.34
    ax.add_patch(plt.Circle((-dx, 0), r, fc='#f39c12', ec='#b9770e', alpha=0.42, lw=2))
    ax.add_patch(plt.Circle(( dx, 0), r, fc='#1abc9c', ec='#0e8f77', alpha=0.42, lw=2))
    ax.text(-dx-0.30, 0, str(c['t_only']), ha='center', va='center', fontsize=13, fontweight='bold')
    ax.text(0, 0, str(c['both']), ha='center', va='center', fontsize=15, fontweight='bold', color='#1d6f42')
    ax.text( dx+0.30, 0, str(c['c_only']), ha='center', va='center', fontsize=13, fontweight='bold')
    ax.text(-dx, 0.70, 'Tracking\ningestion (T)', ha='center', va='bottom', fontsize=9.5,
            color='#b9770e', fontweight='bold')
    ax.text( dx, 0.70, 'Colocalization\nconfirmed (C)', ha='center', va='bottom', fontsize=9.5,
            color='#0e8f77', fontweight='bold')
    pT = 100*c['both']/max(c['T'], 1); pC = 100*c['both']/max(c['C'], 1)
    ax.set_title(f"{title}: double = {c['both']}  ({pT:.0f}% of T, {pC:.0f}% of C)",
                 fontsize=11, fontweight='bold')
    ax.text(0, -0.98, f"neither (contact only): {c['neither']}", ha='center', fontsize=8.5, color='#666')
    ax.set_xlim(-1.35, 1.35); ax.set_ylim(-1.2, 1.3); ax.set_aspect('equal'); ax.axis('off')

if len(combined) and 'coloc_confirmed' in combined.columns:
    combined = categorize(combined)
    groups = [s for s in SIZE_ORDER if s in combined['size'].unique()] + ['all']

    # explicit table: how each row's double / T / C is built (makes the % unambiguous)
    rows = []
    for gname in groups:
        sub = combined if gname == 'all' else combined[combined['size'] == gname]
        c = _venn_counts(sub)
        rows.append({'group': gname, 'contacts_total': len(sub),
                     'tracking_ingestion_T': c['T'], 'coloc_confirmed_C': c['C'],
                     'double_T_and_C': c['both'],
                     'pct_of_T': round(100*c['both']/max(c['T'], 1), 1),
                     'pct_of_C': round(100*c['both']/max(c['C'], 1), 1),
                     'neither': c['neither']})
    corrob_tab = pd.DataFrame(rows)
    print('Two modes as sets -- T = tracking ingestions, C = colocalization-confirmed, '
          'double = T & C (T = tracking_only + double, C = coloc_only + double):')
    display(corrob_tab)

    fig, axes = plt.subplots(2, 2, figsize=(11, 10)); axes = axes.ravel()
    for ax, gname in zip(axes, groups):
        sub = combined if gname == 'all' else combined[combined['size'] == gname]
        _draw_venn(ax, _venn_counts(sub), size_label(gname))
    for ax in axes[len(groups):]:
        ax.axis('off')
    fig.suptitle('Corroboration of the two analysis modes  (orange = tracking ingestion T, teal = colocalization C)',
                 fontweight='bold', y=1.01)
    plt.tight_layout(); plt.savefig(O('phago_corroboration_venn.pdf'), bbox_inches='tight'); plt.show()

    allr = corrob_tab[corrob_tab['group'] == 'all'].iloc[0]
    print(f"Pooled: T = {int(allr.tracking_ingestion_T)} tracking-called ingestions, "
          f"C = {int(allr.coloc_confirmed_C)} colocalization-confirmed contacts; overlap = "
          f"{int(allr.double_T_and_C)} double-corroborated.")
    print(f"  -> the '{allr.pct_of_T:.0f}%' headline is double / T = "
          f"{int(allr.double_T_and_C)} / {int(allr.tracking_ingestion_T)} "
          f"(share of tracking ingestions the pixels confirm); double / C = {allr.pct_of_C:.0f}%.")

    double = combined[combined['category'] == 'double-corroborated'].copy()
    double.to_csv(O('phago_double_corroborated_events.csv'), index=False)
    print(f'{len(double)} double-corroborated events saved to phago_double_corroborated_events.csv')

    # ---- overlay event locations on a representative movie frame (example dataset) ----
    EX = next(((s, r) for (s, r) in per_dataset
               if (categorize(per_dataset[(s, r)]['events'])['category'] == 'double-corroborated').any()),
              list(per_dataset)[0])
    d = per_dataset[EX]; evx = categorize(d['events'])
    rtif, mtif = _find_corrected_tifs(*EX)
    rfh = tifffile.TiffFile(rtif); bfh = tifffile.TiffFile(mtif)
    nfr = min(len(rfh.pages), len(bfh.pages)); midf = nfr // 2
    comp = _composite(rfh.pages[midf].asarray(), bfh.pages[midf].asarray())
    rfh.close(); bfh.close()
    fig, ax = plt.subplots(figsize=(12, 9)); ax.imshow(comp)
    for cat in CAT_ORDER:
        e = evx[evx['category'] == cat]; big = cat == 'double-corroborated'
        ax.scatter(e['x_um']/PIXEL_UM, e['y_um']/PIXEL_UM, s=70 if big else 22,
                   marker='*' if big else 'o', c=CAT_COL[cat],
                   edgecolor='white' if big else 'none', linewidth=0.6,
                   alpha=0.95 if big else 0.5, label=f'{cat} (n={len(e)})', zorder=4 if big else 2)
    ax.set_title(f'{size_label(EX[0])} {EX[1]}: contacts classified by both modes, on the movie (frame {midf+1})\n'
                 f'red=Raji, blue=macrophage', fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([]); ax.legend(loc='upper right', framealpha=0.9, fontsize=9)
    plt.tight_layout(); plt.savefig(O('phago_events_on_image.pdf'), bbox_inches='tight'); plt.show()


## 8d-ii. Phagocytosis vs island size — double-corroborated only

Same comparison as section 8 (panels A/C), but restricted to events where **both** modes
agree (`category == 'double-corroborated'`) — tracking called it an ingestion *and*
colocalization independently confirmed it. This is the strictest evidence of true
engulfment, filtering out tracking-only calls the pixel data doesn't support.


In [ ]:
# ============ phago vs size, restricted to double-corroborated events ============
if len(combined) and 'category' in combined.columns:
    np.random.seed(0)
    sizes_present = [s for s in SIZE_ORDER if s in combined['size'].unique()]
    dbl = combined[combined['category'] == 'double-corroborated']

    cnt_dbl = (combined.groupby(['size', 'replicate'])
               .agg(n_macrophages=('mac_tid', 'nunique'))
               .reset_index())
    dbl_counts = (dbl.groupby(['size', 'replicate']).size()
                  .rename('double_corroborated').reset_index())
    cnt_dbl = cnt_dbl.merge(dbl_counts, on=['size', 'replicate'], how='left')
    cnt_dbl['double_corroborated'] = cnt_dbl['double_corroborated'].fillna(0)
    cnt_dbl['double_per_mac'] = cnt_dbl['double_corroborated'] / cnt_dbl['n_macrophages'].clip(lower=1)

    dbl_by = {s: cnt_dbl[cnt_dbl['size'] == s]['double_corroborated'].values for s in sizes_present}
    dbl_pm_by = {s: cnt_dbl[cnt_dbl['size'] == s]['double_per_mac'].values for s in sizes_present}

    fig, ax = plt.subplots(1, 2, figsize=(13, 5.2))
    violin_scatter(ax[0], dbl_by, 'double-corroborated events / replicate',
                   'E) Double-corroborated phagocytosis vs island size')
    violin_scatter(ax[1], dbl_pm_by, 'double-corroborated events / macrophage',
                   'F) Double-corroborated phagocytosis per macrophage vs island size')
    fig.text(0.5, -0.02, 'double-corroborated = tracking-called ingestion AND colocalization-confirmed; '
             'violin = distribution across replicates; dot = one replicate; black bar = median',
             ha='center', fontsize=9, color='#555')
    plt.tight_layout()
    plt.savefig(O('phago_vs_size_double_corroborated.pdf'), bbox_inches='tight')
    save_panel(fig, ax[0], O('phago_vs_size_double_corroborated_E.pdf'))
    save_panel(fig, ax[1], O('phago_vs_size_double_corroborated_F.pdf'))
    plt.show()
else:
    print('No datasets processed yet, or corroboration not computed - run section 8c/8d first.')


In [ ]:
# --- timing: when double-corroborated ingestion occurs, by size ---
if len(combined) and 'category' in combined.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    for s in sizes_present:
        hr = dbl[dbl['size'] == s]['ingest_hr'].dropna()
        if len(hr):
            ax.hist(hr, bins=np.arange(0, 25, 1), histtype='step', linewidth=2.2,
                    color=SIZE_COLORS.get(s, '#999'), label=size_label(s), density=True)
    ax.set_xlabel('time of ingestion (hours)'); ax.set_ylabel('density')
    ax.set_title('When double-corroborated ingestion occurs, by size', fontweight='bold')
    ax.legend(title='island size'); ax.set_xlim(0, 24); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(O('phago_when_by_size_double_corroborated.pdf'), bbox_inches='tight')
    plt.show()


## 8f. Statistics — phagocytosis vs island size

Three independent island sizes x 3 biological replicates each (n=3/group, unpaired) -
too small to check normality, so we use **Kruskal-Wallis** as the omnibus test across
the three sizes, with an **exact permutation p-value** (scipy's chi-squared
approximation isn't valid at n=3/group; here we enumerate all 1680 ways to relabel the
9 replicates into groups of 3 and compare the observed H to that exact null).

If useful, pairwise exact Mann-Whitney U tests (Benjamini-Hochberg-adjusted for 3
comparisons) are shown as brackets. **Caveat:** with only 3 replicates per group, a
single pairwise comparison has just 20 possible relabelings, so the smallest achievable
exact p-value is 0.10 - no pairwise test can reach p<0.05 at this sample size, however
clean the separation. The omnibus test (which pools all 9 points at once) is what
actually has power here; the pairwise brackets only rank which pairs differ most.

Applied to the two "phagocytosis" metrics (ingestion-confirmed and double-corroborated),
each shown per-replicate and per-macrophage - not to the "contacts" panels, which measure
exposure rather than phagocytosis itself.


In [ ]:
# ============ statistics: phagocytosis vs island size (exact, n=3/group) ============
from itertools import combinations
from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests

def exact_kruskal_pvalue(groups):
    data = np.concatenate(groups)
    sizes = [len(g) for g in groups]
    obs_H, _ = kruskal(*groups)
    idx = list(range(len(data)))
    def partitions(indices, sizes):
        if not sizes:
            yield []
            return
        for combo in combinations(indices, sizes[0]):
            rest = [i for i in indices if i not in combo]
            for other in partitions(rest, sizes[1:]):
                yield [combo] + other
    total = 0; count_ge = 0
    for part in partitions(idx, sizes):
        sample = [data[list(p)] for p in part]
        H, _ = kruskal(*sample)
        total += 1
        if H >= obs_H - 1e-9:
            count_ge += 1
    return obs_H, count_ge / total

def pairwise_stats(groups):
    pairs = [(0, 1), (1, 2), (0, 2)]
    raw_p = [mannwhitneyu(groups[i], groups[j], alternative='two-sided', method='exact')[1]
             for i, j in pairs]
    _, adj_p, _, _ = multipletests(raw_p, method='fdr_bh')
    return pairs, adj_p

def violin_scatter_stats(ax, groups, ylabel, title):
    np.random.seed(0)
    for i, s in enumerate(SIZE_ORDER):
        y = groups[i]
        col = SIZE_COLORS.get(s, '#999')
        if len(y) >= 2 and np.ptp(y) > 0:
            parts = ax.violinplot([y], positions=[i], widths=0.7, showextrema=False)
            for pc in parts['bodies']:
                pc.set_facecolor(col); pc.set_alpha(0.30)
                pc.set_edgecolor(col); pc.set_linewidth(1.3)
        x = np.full(len(y), i) + np.random.uniform(-0.07, 0.07, len(y))
        ax.scatter(x, y, color=col, edgecolor='white', linewidth=0.6, s=75, zorder=3)
        ax.hlines(np.median(y), i - 0.22, i + 0.22, color='black', lw=2.2, zorder=4)
    ax.set_xticks(range(len(SIZE_ORDER))); ax.set_xticklabels([size_label(s) for s in SIZE_ORDER])
    ax.set_ylabel(ylabel); ax.grid(alpha=0.3, axis='y')

    H, p_omni = exact_kruskal_pvalue(groups)
    pairs, adj_p = pairwise_stats(groups)
    ax.set_title(f'{title}\nKruskal-Wallis (exact): H={H:.2f}, p={p_omni:.4f}', fontweight='bold', fontsize=11)

    all_vals = np.concatenate(groups)
    dmax = all_vals.max(); dmin = min(0, all_vals.min())
    span = dmax - dmin if dmax > dmin else 1
    ax.set_ylim(dmin, dmax + span * 0.42)
    step = span * 0.12
    base = dmax + span * 0.06
    for k, ((i, j), p) in enumerate(zip(pairs, adj_p)):
        y = base + k * step
        ax.plot([i, i, j, j], [y, y + step * 0.15, y + step * 0.15, y], color='black', lw=1.1)
        label = f'p={p:.3f}' if p >= 0.001 else 'p<0.001'
        ax.text((i + j) / 2, y + step * 0.22, label, ha='center', va='bottom', fontsize=8.5)

if len(combined) and 'category' in combined.columns:
    stat_summary = summary.merge(
        dbl.groupby(['size', 'replicate']).size().rename('n_double').reset_index(),
        on=['size', 'replicate'], how='left')
    stat_summary['n_double'] = stat_summary['n_double'].fillna(0)
    stat_summary['double_per_mac'] = stat_summary['n_double'] / stat_summary['n_macrophages'].clip(lower=1)

    def groups_for(col):
        return [stat_summary[stat_summary['size'] == s][col].values.astype(float) for s in SIZE_ORDER]

    fig, axes = plt.subplots(2, 2, figsize=(13, 11))
    metrics = [
        (axes[0, 0], 'n_ingested', 'ingestion-confirmed events / replicate',
         'A) Phagocytosis (tracking) vs island size', 'A'),
        (axes[0, 1], 'ingested_per_mac', 'ingestion-confirmed events / macrophage',
         'C) Phagocytosis per macrophage (tracking) vs island size', 'C'),
        (axes[1, 0], 'n_double', 'double-corroborated events / replicate',
         'E) Double-corroborated phagocytosis vs island size', 'E'),
        (axes[1, 1], 'double_per_mac', 'double-corroborated events / macrophage',
         'F) Double-corroborated phagocytosis per macrophage vs island size', 'F'),
    ]
    for ax, col, ylabel, title, tag in metrics:
        violin_scatter_stats(ax, groups_for(col), ylabel, title)

    fig.text(0.5, -0.01, 'violin = distribution across replicates; dot = one replicate; black bar = median; '
             'brackets show pairwise exact Mann-Whitney U, Benjamini-Hochberg-adjusted (n=3/group -> min p=0.10 per pair)',
             ha='center', fontsize=9, color='#555')
    plt.tight_layout()
    plt.savefig(O('phago_vs_size_stats.pdf'), bbox_inches='tight')
    for ax, col, ylabel, title, tag in metrics:
        save_panel(fig, ax, O(f'phago_vs_size_stats_{tag}.pdf'))
    plt.show()
else:
    print('No datasets processed yet, or corroboration not computed - run section 8c/8d first.')


## 8e. Visual proof — clip & TIFF stack of a corroborated event

An in-page animation and a saved multi-frame TIFF (`phago_event_movie.tif`) of one
double-corroborated event: watch the red Raji move into the blue macrophage as M1 climbs
above threshold (green border = colocalized frame). Below it, a montage contrasts one
example of each class (tracking-only, colocalization-only, double-corroborated).

In [ ]:
# ============ in-page clip + TIFF stack + 3-class montage ============
from matplotlib import animation
from IPython.display import HTML

def _crop_series(size, rep, mac_tid, raji_tid, f_lo, f_hi, pad=90):
    d = per_dataset[(size, rep)]; mac = d['mac']; raji = d['raji']
    rtif, mtif = _find_corrected_tifs(size, rep)
    rfh = tifffile.TiffFile(rtif); bfh = tifffile.TiffFile(mtif)
    n = min(len(rfh.pages), len(bfh.pages))
    out = []
    for fn in range(max(0, f_lo), min(n, f_hi + 1)):
        R = rfh.pages[fn].asarray(); B = bfh.pages[fn].asarray()
        ms = mac[(mac['TRACK_ID'] == mac_tid) & (mac['FRAME'] == fn)]
        if len(ms):
            cx = ms['POSITION_X'].iloc[0]/PIXEL_UM; cy = ms['POSITION_Y'].iloc[0]/PIXEL_UM
            rr = (ms['RADIUS'].iloc[0] if pd.notna(ms['RADIUS'].iloc[0]) else 15)/PIXEL_UM
        else:
            cx = cy = None; rr = 15
        comp = _composite(R, B); H, W = comp.shape[:2]
        if cx is None: cx, cy = W/2, H/2
        x0, x1 = max(0, int(cx-pad)), min(W, int(cx+pad))
        y0, y1 = max(0, int(cy-pad)), min(H, int(cy+pad))
        m1v = _manders_m1(R, B, cx, cy, rr)
        rs = raji[(raji['TRACK_ID'] == raji_tid) & (raji['FRAME'] == fn)]
        bp = (rs['POSITION_X'].iloc[0]/PIXEL_UM - x0, rs['POSITION_Y'].iloc[0]/PIXEL_UM - y0) if len(rs) else None
        out.append(dict(frame=fn, crop=comp[y0:y1, x0:x1], mac=(cx-x0, cy-y0, rr), raji=bp, m1=m1v))
    rfh.close(); bfh.close()
    return out

def _scale_bar(ax, img_shape, um=20, frac_w=0.06, frac_h=0.030, pad_frac=0.05):
    """White horizontal scale bar (`um` micrometres) in the bottom-right of an imshow axes."""
    h, w = img_shape[:2]
    bar_px = um / PIXEL_UM                      # 20 um / 0.8 um-per-px = 25 px
    bh = max(2.0, frac_h * h)                   # bar thickness, in image pixels
    pad = pad_frac * w
    x0 = w - pad - bar_px
    y0 = h - pad - bh
    ax.add_patch(plt.Rectangle((x0, y0), bar_px, bh, facecolor='white',
                               edgecolor='none', zorder=5))

def _pick(size, rep, category, lo=5, hi=16):
    ev = categorize(per_dataset[(size, rep)]['events']).copy()
    ev['span'] = ev['end_frame'] - ev['start_frame'] + 1
    c = ev[(ev['category'] == category) & (ev['span'] >= lo) & (ev['span'] <= hi)]
    if not len(c):
        c = ev[ev['category'] == category]
    return c.sort_values('max_coloc_run', ascending=False).iloc[0] if len(c) else None

if len(combined) and 'category' in combined.columns:
    ES = EX  # example dataset from 8d
    e0 = _pick(*ES, 'double-corroborated')
    if e0 is not None:
        ser = _crop_series(*ES, int(e0['mac_tid']), int(e0['raji_tid']),
                            int(e0['start_frame'])-2, int(e0['end_frame'])+2)
        # save TIFF stack (uint8 RGB, frames padded to common size)
        mh = max(s['crop'].shape[0] for s in ser); mw = max(s['crop'].shape[1] for s in ser)
        stack = np.zeros((len(ser), mh, mw, 3), np.uint8)
        for i, s in enumerate(ser):
            c = (np.clip(s['crop'], 0, 1)*255).astype(np.uint8)
            stack[i, :c.shape[0], :c.shape[1]] = c
        tifffile.imwrite(O('phago_event_movie.tif'), stack, photometric='rgb')

        fig, ax = plt.subplots(figsize=(5.4, 5.8))
        def draw(i):
            ax.clear(); s = ser[i]; ax.imshow(s['crop'])
            cx, cy, rr = s['mac']; ax.add_patch(plt.Circle((cx, cy), rr, fill=False, ec='cyan', lw=2))
            if s['raji']: ax.plot(*s['raji'], 'x', color='yellow', ms=13, mew=2.5)
            good = (not np.isnan(s['m1'])) and s['m1'] >= COLOC_M1_MIN
            col = '#2ecc71' if good else '#f39c12'
            ax.set_title(f"frame {s['frame']+1}   t={s['frame']*MIN_PER_FRAME/60:.1f} h   "
                         f"M1={s['m1']:.2f}{'  COLOC' if good else ''}", color=col, fontweight='bold')
            for sp in ax.spines.values(): sp.set_color(col); sp.set_linewidth(5)
            ax.set_xticks([]); ax.set_yticks([])
        anim = animation.FuncAnimation(fig, draw, frames=len(ser), interval=550)
        html = anim.to_jshtml(); plt.close(fig)
        print(f'Double-corroborated example: {ES[0]} {ES[1]}  mac#{int(e0["mac_tid"])} '
              f'B#{int(e0["raji_tid"])}  frames {ser[0]["frame"]+1}-{ser[-1]["frame"]+1}  '
              f'(cyan o = macrophage, yellow x = Raji);  saved phago_event_movie.tif')
        display(HTML(html))

    # ---- 3-class montage: one example event per class, 3 frames each ----
    classes = ['tracking-only', 'coloc-only', 'double-corroborated']
    picks = [(c, _pick(*ES, c)) for c in classes]
    picks = [(c, e) for c, e in picks if e is not None]
    if picks:
        fig, axes = plt.subplots(len(picks), 3, figsize=(11, 3.7*len(picks)), squeeze=False)
        for r, (cat, e) in enumerate(picks):
            ser = _crop_series(*ES, int(e['mac_tid']), int(e['raji_tid']),
                               int(e['start_frame']), int(e['end_frame']))
            idxs = sorted(set([0, len(ser)//2, len(ser)-1]))
            while len(idxs) < 3: idxs.append(len(ser)-1)
            for c, k in enumerate(idxs[:3]):
                ax = axes[r][c]; s = ser[k]; ax.imshow(s['crop'])
                cx, cy, rr = s['mac']; ax.add_patch(plt.Circle((cx, cy), rr, fill=False, ec='cyan', lw=1.8))
                if s['raji']: ax.plot(*s['raji'], 'x', color='yellow', ms=11, mew=2, alpha=0.75)
                _scale_bar(ax, s['crop'].shape, um=20)
                ax.set_title(f"frame {s['frame']+1}  M1={s['m1']:.2f}", fontsize=9)
                ax.set_xticks([]); ax.set_yticks([])
            axes[r][0].set_ylabel(cat, color=CAT_COL[cat], fontweight='bold', fontsize=12,
                                  rotation=0, ha='right', va='center', labelpad=40)
        fig.suptitle(f'{size_label(ES[0])} {ES[1]}: one event per class  (red=Raji, blue=macrophage, cyan o=mac, yellow x=Raji)',
                     fontweight='bold', y=1.01)
        plt.tight_layout(); plt.savefig(O('phago_corroboration_montage.pdf'), bbox_inches='tight'); plt.show()


## 9. Notes — methods, parameters, outputs

**Inputs (ground truth).** Cellpose segmentation + TrackMate LAP tracking (drift-corrected
in Fiji). Positions in microns, time in seconds; `PIXEL_UM` and `MIN_PER_FRAME` convert to
px / minutes. Cell type is inferred per file by median spot **radius** (Raji ~6 um,
macrophage ~13 um), not by channel number, because channel numbering is inconsistent across
acquisitions.

**Mode 1 — tracking (geometric).** `find_contacts` caches every Raji-macrophage proximity
at `FACTOR_MAX`; `filter_contacts` applies `RADIUS_FACTOR`; `build_events` keeps runs of
>= `MIN_FRAMES` frames (gap `GAP_TOL`) and flags **ingestion** when the Raji track ends
inside the overlap (+ `END_TOL`) and before the last frame. Robust metric = ingestion count;
raw contact count is sensitive to `RADIUS_FACTOR`.

**Mode 2 — colocalization (pixel).** `colocalize_dataset` runs on the drift-corrected
movies over the *same candidate contacts*. Per contact frame: Manders' **M1** = fraction of
local Raji (red) signal inside the Otsu-masked macrophage (blue) body, within an ROI of
`COLOC_PAD` x macrophage radius. Frame colocalized if M1 >= `COLOC_M1_MIN`; event
**colocalization-confirmed** if sustained >= `MIN_FRAMES` (gap `GAP_TOL`). Frames are read
lazily (per-dataset cache).

**Corroboration (8d).** category = double-corroborated / tracking-only / colocalization-only
/ contact. Double-corroborated = both modes agree = highest confidence.

**Key parameters (cell 2, edit there):** `PIXEL_UM`, `MIN_PER_FRAME`, `RADIUS_FACTOR`,
`MIN_FRAMES`, `GAP_TOL`, `END_TOL`, `FACTOR_MAX`, `RAJI_MAX_RADIUS_UM`; colocalization knobs
`COLOC_M1_MIN`, `COLOC_PAD` (cell 8c).

**Outputs written** — all into `OUTDIR = outputs/<YYYYMMDD>_trial<N>/` (set `TRIAL` in cell 2; `None` = next unused for today)**:** `phago_vs_size.png`, `phago_when_by_size.png`, `phago_where_by_size.png`,
`phago_where_by_phase.png`, `phago_colocalization_validation.{png,csv}`,
`phago_double_corroborated_events.csv`, `phago_corroboration_venn.png`, `phago_events_on_image.png`, `phago_event_movie.tif`,
`phago_corroboration_montage.png`, plus the per-dataset / manual-check exports.

**Caveats.** (1) Colocalization assumes the two channels share the same drift-corrected pixel
grid; an independent per-channel registration offset would depress M1 roughly uniformly.
(2) M1 uses an Otsu macrophage mask and a 20th-percentile local red-background subtraction;
`COLOC_M1_MIN` and `COLOC_PAD` can be swept like the tracking thresholds. (3) Colocalization
is evaluated only on tracking's candidate contacts, so a truly track-free engulfment is not
detected.

## 10. ✏️ Retuning & threshold sweeps

Two ways to re-tune **without touching cell 2** (contacts are cached, so both are instant):

**(a) Live re-tune** — pick one parameter set and refresh every table/plot. After running
`retune(...)`, re-run §6–§8 to see the updated figures.

**(b) Threshold sweep** — scan a grid of `RADIUS_FACTOR × MIN_FRAMES` and see how event and
ingestion counts respond. Edit `SWEEP_FACTORS` / `SWEEP_MINFRAMES` below (all factors must be
≤ `FACTOR_MAX`).

In [ ]:
def retune(radius_factor=1.0, min_frames=6, gap_tol=1, end_tol=2):
    # rebuild `combined` and each dataset's events with new params (from cached contacts)
    global combined
    combined = process_all(radius_factor, min_frames, gap_tol, end_tol, verbose=True)
    print(f'\nretuned: factxxor={radius_factor}, min_frames={min_frames}, '
          f'gap_tol={gap_tol}, end_tol={end_tol} -> {len(combined)} events, '
          f'{int(combined["ingested"].sum()) if len(combined) else 0} ingested')
    return combined

# ===================== ✏️ EDIT: live re-tune (then re-run §6-§8) =====================
# combined = retune(radius_factor=0.75, min_frames=4)
# =====================================================================================

In [ ]:
def sweep_dataset(key, factors, minframes, gap_tol=GAP_TOL, end_tol=END_TOL):
    d = per_dataset[key]
    rows = []
    for fac in factors:
        fc = filter_contacts(d['contacts'], fac)
        for mf in minframes:
            ev = build_events(fc, d['raji'], mf, gap_tol, end_tol)
            rows.append({'radius_factor': fac, 'min_frames': mf,
                         'n_events': len(ev),
                         'n_ingested': int(ev['ingested'].sum()) if len(ev) else 0})
    return pd.DataFrame(rows)


def plot_sweep(key, sweep):
    ev_grid = sweep.pivot(index='radius_factor', columns='min_frames', values='n_events')
    in_grid = sweep.pivot(index='radius_factor', columns='min_frames', values='n_ingested')
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    for ax, grid, title, cmap in [(axes[0], ev_grid, 'events', 'Blues'),
                                  (axes[1], in_grid, 'ingestion-confirmed', 'Reds')]:
        im = ax.imshow(grid.values, cmap=cmap, aspect='auto', origin='lower')
        ax.set_xticks(range(len(grid.columns))); ax.set_xticklabels(grid.columns)
        ax.set_yticks(range(len(grid.index))); ax.set_yticklabels(grid.index)
        ax.set_xlabel('MIN_FRAMES'); ax.set_ylabel('RADIUS_FACTOR')
        ax.set_title(f'{size_label(key[0])} {key[1]}: {title}', fontweight='bold')
        for i in range(grid.shape[0]):
            for j in range(grid.shape[1]):
                ax.text(j, i, int(grid.values[i, j]), ha='center', va='center', fontsize=9)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show()

# ===================== ✏️ EDIT: sweep grid (factors must be <= FACTOR_MAX) =====================
SWEEP_FACTORS   = [0.5, 0.75, 1.0, 1.25, 1.5]
SWEEP_MINFRAMES = [2, 3, 4, 6]
# =============================================================================================

# --- commented out: parameter sweeps are time-consuming and not needed for the publication figure
#     run; uncomment when validating RADIUS_FACTOR / MIN_FRAMES again ---
# for key in per_dataset:
#     sw = sweep_dataset(key, SWEEP_FACTORS, SWEEP_MINFRAMES)
#     sw.to_csv(O(f'phago_sweep_{key[0]}_{key[1]}.csv'), index=False)
#     print(str(key) + ': sweep saved to ' + O(f'phago_sweep_{key[0]}_{key[1]}.csv'))
#     plot_sweep(key, sw)


## 10b. Colocalization parameter sweep

Sensitivity of the colocalization confirmation to its two knobs — the M1 threshold
`COLOC_M1_MIN` and the ROI size `COLOC_PAD`. For each dataset the per-frame Manders' M1 is
recomputed at every `COLOC_PAD` (decoded frames are reused), then confirmation is re-scored
across a grid of `COLOC_M1_MIN` cheaply. Heatmaps show, per island size / replicate, the %
of tracking-ingestion events confirmed and the number of double-corroborated events, so you
can choose thresholds the way §10 does for tracking. (`MIN_FRAMES` / `GAP_TOL` for the
sustained run are shared with the tracking limits.)

In [ ]:
# ============ colocalization parameter sweep (COLOC_M1_MIN x COLOC_PAD) ============
def coloc_sweep_dataset(key, m1_mins, pads, min_frames=None, gap_tol=None):
    mf = MIN_FRAMES if min_frames is None else min_frames
    gt = GAP_TOL if gap_tol is None else gap_tol
    d = per_dataset.get(key)
    if d is None or 'events' not in d or len(d['events']) == 0:
        return pd.DataFrame()
    rtif, mtif = _find_corrected_tifs(*key)
    if rtif is None or mtif is None:
        print(f'  {key}: tif(s) missing - skipped'); return pd.DataFrame()
    mac = d['mac']; lut = {}
    for tid, fr, x, y, r in zip(mac['TRACK_ID'], mac['FRAME'], mac['POSITION_X'],
                                mac['POSITION_Y'], mac['RADIUS']):
        lut[(int(tid), int(fr))] = (x/PIXEL_UM, y/PIXEL_UM, (r if pd.notna(r) else 12)/PIXEL_UM)
    rfh = tifffile.TiffFile(rtif); bfh = tifffile.TiffFile(mtif)
    nred, nblue = len(rfh.pages), len(bfh.pages); fcache = {}
    def gf(tf, k, i, n):
        if i < 0 or i >= n: return None
        kk = (k, i)
        if kk not in fcache: fcache[kk] = tf.pages[i].asarray()
        return fcache[kk]
    # per event: ingested flag + {pad: [(frame, m1), ...]}   (decode each frame once, all pads)
    ev_series = []
    for e in d['events'].itertuples():
        perpad = {p: [] for p in pads}
        for fn in range(int(e.start_frame), int(e.end_frame) + 1):
            p0 = lut.get((int(e.mac_tid), fn))
            if p0 is None: continue
            R = gf(rfh, 'r', fn, nred); B = gf(bfh, 'b', fn, nblue)
            if R is None or B is None: continue
            for pad in pads:
                perpad[pad].append((fn, _manders_m1(R, B, p0[0], p0[1], p0[2], pad)))
        ev_series.append((bool(e.ingested), perpad))
    rfh.close(); bfh.close()
    n_ing = sum(1 for ing, _ in ev_series if ing)
    rows = []
    for pad in pads:
        for t in m1_mins:
            n_ci = n_call = 0
            for ing, perpad in ev_series:
                frames = [fn for fn, m1 in perpad[pad] if (m1 == m1 and m1 >= t)]
                conf = _max_sustained(frames, mf, gt) >= mf
                n_call += conf
                n_ci += (conf and ing)
            rows.append({'pad': pad, 'm1_min': t, 'n_ingest': n_ing, 'n_double': n_ci,
                         'coloc_rate_%': round(100*n_ci/max(n_ing, 1), 1),
                         'n_coloc_contacts': n_call})
    return pd.DataFrame(rows)


def plot_coloc_sweep(key, sweep):
    rate = sweep.pivot(index='pad', columns='m1_min', values='coloc_rate_%')
    dbl = sweep.pivot(index='pad', columns='m1_min', values='n_double')
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
    for ax, grid, title, cmap, fmt in [(axes[0], rate, 'ingestion confirmation rate (%)', 'Greens', '{:.0f}'),
                                       (axes[1], dbl, 'double-corroborated events', 'Purples', '{:d}')]:
        im = ax.imshow(grid.values, cmap=cmap, aspect='auto', origin='lower')
        ax.set_xticks(range(len(grid.columns))); ax.set_xticklabels(grid.columns)
        ax.set_yticks(range(len(grid.index))); ax.set_yticklabels(grid.index)
        ax.set_xlabel('COLOC_M1_MIN'); ax.set_ylabel('COLOC_PAD')
        ax.set_title(f'{size_label(key[0])} {key[1]}: {title}', fontweight='bold')
        for i in range(grid.shape[0]):
            for j in range(grid.shape[1]):
                v = grid.values[i, j]
                ax.text(j, i, fmt.format(int(v) if fmt == '{:d}' else v),
                        ha='center', va='center', fontsize=9)
        # mark the current operating point
        if COLOC_PAD in list(grid.index) and COLOC_M1_MIN in list(grid.columns):
            ax.add_patch(Rectangle((list(grid.columns).index(COLOC_M1_MIN)-0.5,
                                    list(grid.index).index(COLOC_PAD)-0.5), 1, 1,
                                   fill=False, ec='red', lw=2.5))
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show()

# ===================== EDIT: colocalization sweep grid =====================
COLOC_SWEEP_M1   = [0.3, 0.4, 0.5, 0.6, 0.7]
COLOC_SWEEP_PADS = [1.0, 1.5, 2.0]   # red box on heatmaps marks current COLOC_M1_MIN x COLOC_PAD
# ===========================================================================

# --- commented out: colocalization sweep re-decodes every frame per dataset per (pad, m1_min)
#     combo and is the slowest cell in the notebook; not needed for the publication figure run.
#     Uncomment when re-validating COLOC_M1_MIN / COLOC_PAD. ---
# if len(combined):
#     coloc_sweeps = {}
#     for key in per_dataset:
#         sw = coloc_sweep_dataset(key, COLOC_SWEEP_M1, COLOC_SWEEP_PADS)
#         if len(sw):
#             coloc_sweeps[key] = sw
#             sw.to_csv(O(f'phago_colocsweep_{key[0]}_{key[1]}.csv'), index=False)
#             print(str(key) + ': colocalization sweep saved to ' + O(f'phago_colocsweep_{key[0]}_{key[1]}.csv'))
#             plot_coloc_sweep(key, sw)


### 11. ✏️ Events to manually check in the video (top 3 & bottom 3)

Ranks the phagocytic events and lists the **top `N`** and **bottom `N`** so you can eyeball them in
the movie. For each event you get everything needed to navigate in **Fiji/ImageJ** on the
**drift-corrected** stack (`*_xyCorrected.tif`):

- **WHEN** — `start_frame_1idx` / `end_frame_1idx` (**1-indexed**, as ImageJ shows) and hours;
- **WHERE** — overlap location in **pixels** (`overlap_x_px`, `overlap_y_px`) and µm, plus the
  B-cell's **origin** in pixels;
- **IDs** — `mac_track` and `bcell_track` to locate the tracks back in TrackMate.

Ranking metric is `RANK_BY` (default `duration_min` = how long the overlap persists). Set
`CONFIRMED_ONLY=False` to rank over all contacts, or change `N_CHECK`.

In [ ]:
# ================= EDIT: manual-check settings =================
FOCUS          = ('60um', 'R5') # (size, replicate) to inspect; None = rank across all datasets pooled
N_CHECK        = 3              # how many from the top and from the bottom
RANK_BY        = 'duration_min' # 'duration_min' or 'n_contact_frames'
CONFIRMED_ONLY = True           # rank only ingestion-confirmed events
# ==================================================================

def focus_df(df):
    if FOCUS is None:
        return df
    return df[(df['size'] == FOCUS[0]) & (df['replicate'] == FOCUS[1])]

def events_to_check(df, n=N_CHECK, by=RANK_BY, confirmed_only=CONFIRMED_ONLY):
    d = df[df['ingested']] if confirmed_only else df
    d = d.dropna(subset=[by]).sort_values(by, ascending=False).reset_index(drop=True)
    if len(d) == 0:
        return pd.DataFrame()
    n = min(n, len(d))
    picks = pd.concat([d.head(n).assign(rank_group='TOP'),
                       d.tail(n).assign(rank_group='BOTTOM')])
    out = pd.DataFrame({
        'rank_group':        picks['rank_group'],
        'size':              picks['size'],
        'replicate':         picks['replicate'],
        by:                  picks[by],
        'mac_track':         picks['mac_tid'],
        'bcell_track':       picks['raji_tid'],
        'start_frame_1idx':  picks['start_frame'] + 1,   # ImageJ is 1-indexed
        'end_frame_1idx':    picks['end_frame'] + 1,
        'start_hr':          (picks['start_frame'] * MIN_PER_FRAME / 60).round(2),
        'end_hr':            (picks['end_frame'] * MIN_PER_FRAME / 60).round(2),
        'ingest_frame_1idx': (picks['raji_last_frame'] + 1),
        'overlap_x_px':      (picks['x_um'] / PIXEL_UM).round().astype(int),
        'overlap_y_px':      (picks['y_um'] / PIXEL_UM).round().astype(int),
        'overlap_x_um':      picks['x_um'],
        'overlap_y_um':      picks['y_um'],
        'bcell_origin_x_px': (picks['orig_x'] / PIXEL_UM).round().astype('Int64'),
        'bcell_origin_y_px': (picks['orig_y'] / PIXEL_UM).round().astype('Int64'),
    })
    return out.reset_index(drop=True)

if len(combined):
    check = events_to_check(focus_df(combined))
    check.to_csv(O('phago_manual_check_top_bottom.csv'), index=False)
    scope = 'all datasets' if FOCUS is None else f'{FOCUS[0]} {FOCUS[1]}'
    print(f'Top {N_CHECK} and bottom {N_CHECK} phagocytic events by {RANK_BY} for {scope} '
          f'(check on the *_xyCorrected.tif stack, frames 1-indexed):')
    display(check)
else:
    print('No datasets processed yet - fill in DATASETS (cell 2).')


## 12. ✏️ Visual snapshots — XY + frame overlaid on the video

For each event in the manual-check list (§11), this pulls the actual **drift-corrected** frames and
shows a **composite** (red = Raji B cell, blue = macrophage) cropped around the event, with:
- the **frame number (1-indexed) and time** in each panel title,
- a **cyan circle** on the macrophage and a **yellow ✕** on the B cell (positions looked up from
  the tracks at that exact frame),
- the absolute **pixel (X, Y)** to type into ImageJ, in the row label.

Three frames are shown per event (**start / middle / end** of the overlap). It auto-finds the
`C=1` and `C=2` `*xyCorrected*.tif` in each dataset's folder. Set `save=True` to write PNGs.

In [ ]:
import tifffile
_TIF_CACHE = {}

def _find_corrected_tifs(size, rep):
    # macrophage movie = the radius-resolved bg tif; Raji movie = the other xyCorrected tif.
    # (channel naming varies: C=1/C=2, C2-/C3-, C=1.tif ... so we don't hardcode tokens)
    entry = DATASETS.get(size, {}).get(rep)
    if not isinstance(entry, str):
        return None, None
    folder = entry if os.path.isabs(entry) else os.path.join(BASE, entry)
    tifs = sorted(f for f in glob.glob(os.path.join(folder, '*xyCorrected*.tif'))
                  if 'copy' not in os.path.basename(f).lower())
    mtif = per_dataset.get((size, rep), {}).get('bg')
    if mtif not in tifs:
        mtif = tifs[-1] if tifs else None
    rtif = next((t for t in tifs if t != mtif), None)
    return rtif, mtif

def _stack(path):
    if path is None:
        return None
    if path not in _TIF_CACHE:
        _TIF_CACHE[path] = tifffile.imread(path)
    return _TIF_CACHE[path]

def _stretch(a):
    a = a.astype(float)
    lo, hi = np.percentile(a, [2, 99])
    return np.clip((a - lo) / (hi - lo + 1e-8), 0, 1)

def _composite(rstack, mstack, frame):
    ref = mstack if mstack is not None else rstack
    H, W = ref[frame].shape[-2], ref[frame].shape[-1]
    comp = np.zeros((H, W, 3))
    if rstack is not None: comp[..., 0] = _stretch(rstack[frame])   # red = Raji
    if mstack is not None: comp[..., 2] = _stretch(mstack[frame])   # blue = macrophage
    return comp

def _pos_px(spots, tid, frame):
    s = spots[(spots['TRACK_ID'] == tid) & (spots['FRAME'] == frame)]
    if len(s):
        return s['POSITION_X'].iloc[0] / PIXEL_UM, s['POSITION_Y'].iloc[0] / PIXEL_UM
    return None

def show_event_snapshots(check=None, pad_px=110, save=False):
    if check is None:
        check = events_to_check(focus_df(combined))
    if len(check) == 0:
        print('no events to show'); return
    ncols = 3
    fig, axes = plt.subplots(len(check), ncols, figsize=(4.2*ncols, 4.2*len(check)), squeeze=False)
    for r, (_, ev) in enumerate(check.iterrows()):
        size, rep = ev['size'], ev['replicate']
        rtif, mtif = _find_corrected_tifs(size, rep)
        rstack, mstack = _stack(rtif), _stack(mtif)
        d = per_dataset.get((size, rep), {})
        f0, f1 = ev['start_frame_1idx'] - 1, ev['end_frame_1idx'] - 1   # back to 0-index
        frames = sorted(set([f0, (f0 + f1) // 2, f1]))
        while len(frames) < ncols:
            frames.append(f1)
        for c in range(ncols):
            ax = axes[r][c]; fr = frames[c]
            if mstack is None and rstack is None:
                ax.text(0.5, 0.5, f'{size_label(size)} {rep}\n(tif not found)', ha='center'); ax.axis('off'); continue
            comp = _composite(rstack, mstack, fr)
            # centre crop on the macrophage position at this frame (fallback: overlap px)
            mp = _pos_px(d.get('mac'), ev['mac_track'], fr) if 'mac' in d else None
            cx, cy = mp if mp else (ev['overlap_x_px'], ev['overlap_y_px'])
            H, W = comp.shape[:2]
            x0, x1 = max(0, int(cx-pad_px)), min(W, int(cx+pad_px))
            y0, y1 = max(0, int(cy-pad_px)), min(H, int(cy+pad_px))
            ax.imshow(comp[y0:y1, x0:x1])
            if mp: ax.plot(cx-x0, cy-y0, 'o', mfc='none', mec='cyan', ms=22, mew=2)
            bp = _pos_px(d.get('raji'), ev['bcell_track'], fr) if 'raji' in d else None
            if bp and x0 <= bp[0] < x1 and y0 <= bp[1] < y1:
                ax.plot(bp[0]-x0, bp[1]-y0, 'x', color='yellow', ms=12, mew=2.5)
            tag = ' (ingest)' if fr == ev['ingest_frame_1idx'] - 1 else ''
            ax.set_title(f'frame {fr+1} - {fr*MIN_PER_FRAME/60:.2f} h{tag}', fontsize=10)
            ax.set_xticks([]); ax.set_yticks([])
        axes[r][0].set_ylabel(f"{ev['rank_group']}  {size_label(size)} {rep}\nmac#{ev['mac_track']} "
                              f"B#{ev['bcell_track']}\n@({ev['overlap_x_px']},{ev['overlap_y_px']}) px",
                              fontsize=9, rotation=0, ha='right', va='center', labelpad=45)
    scope = 'all' if FOCUS is None else f'{size_label(FOCUS[0])} {FOCUS[1]}'
    fig.suptitle(f'Phagocytic events [{scope}] - red=Raji, blue=macrophage, cyan o=mac, yellow x=B cell',
                 fontsize=12, fontweight='bold', y=1.005)
    plt.tight_layout()
    if save:
        plt.savefig(O('phago_event_snapshots.pdf'), bbox_inches='tight')
    plt.show()

if len(combined):
    show_event_snapshots(save=True)
